# Turning 500 GB of DICOM into 11 GiB — and decoding it without a GPU

**Credit.** This builds directly on public work by
[**Pilkwang Kim**](https://www.kaggle.com/code/pilkwang/rsna-knee-baseline-v1)
(`rsna-knee-baseline-v1` — header-recovered slot scheme),
[**Karnakbayev Artur**](https://www.kaggle.com/code/karnakbayevartur/rsna-knee-eda-to-2-5d)
(`rsna-knee-eda-to-2-5d` — the 2.5D framing and the series-metadata EDA), and
[**Will / wguesdon**](https://www.kaggle.com/code/wguesdon/rsna-knee-dinov2-at-meniscus-resolution)
(`rsna-knee-dinov2-at-meniscus-resolution` — millimetres per DINOv2 patch token, and
ordering slices by physical position rather than filename). Thanks also to
**Alexandre Moritz** and **Roman Rozen**. The decode pipeline below is a synthesis of
those notebooks; what is new here is persisting its output and measuring the cost.

---

## The one-sentence version

The model never sees 500 GB. It sees a **4,407 x 6 x 9 x 224 x 224 uint8** array — about
**11.1 GiB** — and every run we have ever done rebuilt that array from DICOM and then threw
it away. This notebook builds it once, writes it out, and measures what that costs.

## Why this matters right now

Kaggle's weekly GPU quota is 30 hours, and by the back half of a competition week a lot of
us have spent it. **The decode is not GPU work.** It is DICOM reads and resizes. So it runs
on a CPU kernel, which draws nothing from that quota — and once the cache exists, a
training run starts at the model instead of spending the first stretch on pixels.


## Measured on the full corpus — one CPU kernel, start to finish

Every figure below is printed by the code in this notebook, on all **4,407** training
studies. Nothing here is projected.

| | |
|---|---|
| Wall clock, whole job | **64.2 min** |
| Written | **11.13 GiB** (4 shards x 2.78 GiB) |
| Slices read | **207,801** |
| Slot-series decoded | **23,089** |
| Sustained decode rate | **~6.25 slot-series/s** |
| Header pass | 110 s |
| Slice ordering resolved by geometry | **100%** |
| Decode failures | **0** |
| MONOCHROME1 series needing inversion | **0** |
| Mean slot coverage | **87.3%** (5.24 of 6 slots per study) |

**CPU is not slower than the accelerator machine.** Our GPU-machine runs put the same
decode at roughly 55 minutes; this CPU-only kernel did the whole corpus in 64. That is the
tell that this stage never used the accelerator at all — which is exactly why it can be
moved off it, and why it runs while your GPU quota is at zero.

### A correction to our own estimate, kept in

A 40-study probe measured **9.0 slot-series/s** and projected **0.71 h**. The real
sustained rate was **6.25/s** and the decode took **1.03 h** — the probe was **44%
optimistic**. Small samples draw thin series; the projection is a go/no-go signal, not a
schedule. We are leaving the miss in the notebook because we made the same class of error
before, in the other direction, estimating a CPU job at 8.66 h that took 2 h.

**And the reason it was slow was not the reason we guessed.** We assumed writing 11 GiB
would dominate. It did not, and it is not close: each 2.78 GiB shard wrote in **3.5–9.4
seconds** (~470 MB/s), about **0.6%** of total runtime. This job is decode-bound end to
end. Worth knowing before you optimise the wrong half — as we nearly did.

## A second number worth more than the first

The pipeline crops each slice to a fixed **physical** extent (`crop_mm`) so a meniscus is
the same size in pixels regardless of the acquisition's field of view. The crop guard
reduces algebraically to `FOV > crop_mm` — so any series at or below `crop_mm` gets **no
physical normalisation at all**.

The measured *median* field of view in this corpus is **160 mm**. With `crop_mm=160`, the
threshold sits exactly at the median:

| `crop_mm` | physical crop applied | skipped |
|---|---|---|
| 160 | 38.9% | **61.1%** |
| **130** | **98.5%** | 1.5% |

*(Both rows measured on this corpus; the 130 row is the full 23,089 slot-series decoded by
the run below.)*

Standing at the median disables your own normalisation on most of the corpus, by
construction. Shrinking the crop is not only a finer pixel pitch — it is the difference
between normalising a third of the data and normalising nearly all of it.

## Two traps we paid for, so you don't have to

**1. Injected code silently shadows the pipeline.** Our generated cell is appended last, so
a top-level name in it *replaces* the pipeline's. A helper called `probe` collided with the
header-pass worker `probe(item)`; the run died 90 seconds in with an arity error inside a
thread pool, naming the symptom and not one word of the cause. If you generate notebooks,
diff your injected names against the host namespace before you push.

**2. A completeness check can be fooled by its own arithmetic.** The run below reported
`INCOMPLETE, still missing: test.s00of04` while having decoded everything correctly: the
test split holds 3 placeholder studies cut into 4 blocks, so one block is empty *by
construction*. An empty shard is vacuously complete, not missing. Fixed — but the guard
firing on a non-problem is far better than the alternative, and it is the reason we trust
it on the shards that matter.

**3. Out-of-fold CV is structurally blind to ensembling.** In OOF each study is predicted by
exactly **one** model — the fold that did not train on it. In a submission **every** member
predicts **every** test study and the lot are rank-meaned. So OOF measures a single model
while the LB measures an ensemble. We measured the gap at **+0.024 with 5 members and
+0.031 with 10**. Our OOF rated a blend at +0.0002 over its best leg, band including zero;
the leaderboard paid **+0.006**. Every "ensembling adds nothing" reading taken off an OOF
number is measuring the wrong object.

## Open proposal, with no result attached: RAD-DINO

**This section is a hypothesis. No fold has run. Do not read it as a finding.**

Most public solutions here, ours included, use **DINOv2** — pretrained on natural images.
[**RAD-DINO**](https://huggingface.co/microsoft/rad-dino) (Microsoft, MIT licence) is the
same DINOv2 architecture finetuned on **882,775 chest X-rays**
([paper](https://huggingface.co/papers/2401.10815)). Same body, medical eyes. We could not
find it discussed anywhere in this competition.

**The hypothesis:** an encoder pretrained on chest radiographs transfers to knee MRI better
than one pretrained on photographs.

**The counter-hypothesis, which we think is at least as likely:** chest X-ray is
transmission through bone and knee MRI is proton signal in soft tissue. A fluid-bright
sequence looks nothing like a chest film. "Medical" is not one domain, and RAD-DINO may
have specialised itself *away* from general features without gaining anything for knees.

**The confound to avoid.** RAD-DINO is ViT-**Base**. If your baseline runs `dinov2-small`,
swapping straight to it moves **two** levers — domain *and* capacity — and a win tells you
nothing about which one won. The clean design is three arms: `small`, `base`, `rad-dino`.
`small → base` prices capacity; `base → rad-dino` prices medical pretraining at matched
size.

**Recorded before the number exists:** we predict RAD-DINO wins, on the grounds that it is
human anatomy and radiological. We are publishing that prediction now precisely so it can
be wrong in public. If someone runs it before we do, please post the fold.

---

## The code

Below is the working decoder. It is sharded and resumable — a shard already on disk is
skipped, so a kernel that dies at hour 11 does not cost the hours before it — and it
refuses to write a **truncated** shard, because a partially-decoded cache does not raise
when you train on it; the missing studies simply read as absent slots. It also writes a
`cache_meta.json` fingerprint (`img`, `crop_mm`, window, slot order, seed) so a training
run can refuse a cache that does not match its own config.

Set the settings cell to whatever your own pipeline uses. **A cache built at different
settings than your model expects is silently wrong, not loudly wrong.**


## 0. Run settings

Kaggle has no way to pass environment variables into a notebook, so the knobs are set
here instead — this is the only cell meant to be edited.

**`RSNA_FOLDS` ships at `1` on purpose.** A full five-fold run is roughly 6 hours of
GPU, and a run that dies at hour five on some data quirk costs a large slice of the
weekly quota for nothing. Prove it works on one fold first, then raise it.

**`RSNA_TRAIN_SEED` is deliberately separate from `CFG.seed`.** It drives only head
init, batch order, group choice and augmentation. Folds come from a hash of the report
and the derived targets come from `CFG.seed`, so both stay byte-identical when this
one moves. Run the same config twice with two values and the gap between the two OOF
scores is the noise floor — the margin any later change has to beat before its result
means anything.

In [1]:
import os

os.environ["RSNA_CROP_MM"] = "130"
os.environ["RSNA_N_GROUP_MAX"] = "3"
os.environ["RSNA_WINDOW"] = "0.35,0.65"

for k, v in os.environ.items():
    if k.startswith("RSNA_"):
        print(f"{k} = {v}")

RSNA_CROP_MM = 130
RSNA_N_GROUP_MAX = 3
RSNA_WINDOW = 0.35,0.65


## 2. Where the targets come from

`train.csv` has a `Report` column and `test.csv` does not. Text is available when
fitting and absent when predicting, which rules out any fusion model with a text branch
and leaves the reports usable only as a source of training targets and of sample
weights.

Two facts shape the extractor:

**Reports are graded, annotations are thresholded.** The reporting radiologist and the
annotator do not share a cutoff — a report saying *small joint effusion* can sit against
a negative annotation. So a rule of the form *term present ⇒ positive* is wrong by
construction; grading the mention (trace / unqualified / marked) is right, and costs
nothing, because §1 established that only order is read.

**Silence is not a negative.** A finding the report never mentions gets a low score and
a low *confidence*, and the confidence becomes the sample weight. A study whose report
says nothing about synovitis pulls on the synovitis head far less than one that names it.

In [2]:
"""Report -> twelve graded targets, in nine languages.

Provenance: the multilingual lexicon and the assert/negate/hedge scoring are adapted
from the community notebook `rsna-knee-baseline-v1`. What is added here is §2 of this
module - `distill`, which fits a text model to the *rule* scores out-of-fold and blends
it back in.

That addition exists because of a failure mode the rule extractor cannot fix from
inside. A rule that never fires does not raise; it emits a negative. A lexicon that is
thick in English and thin in Greek therefore looks like a corpus where Greek patients
have fewer findings, and because language tracks the reporting site, that bias is
aligned with a scanner and a population rather than averaging out as noise.

A character n-gram model fitted to the rule scores over the *whole* corpus sees the
Greek phrasings that co-occur with rule-positive reports and scores them, without anyone
having written them into the lexicon. Fitted out-of-fold and grouped on report text, it
cannot simply memorise the rules it was trained on.
"""

from __future__ import annotations

import re
import unicodedata

import numpy as np

TARGETS = [
    "ACL", "MCL", "Medial Meniscus", "Lateral Meniscus",
    "Medial OA", "Lateral OA", "PF OA", "Effusion",
    "Synovitis", "Baker's", "Contusion", "Fracture",
]

In [3]:
# Turkish dotted/dotless i must be folded before casefolding, otherwise "İZLENMEZ"
# and "izlenmez" diverge. ss and the Croatian/Serbian d-with-stroke likewise.
_PRE = str.maketrans({
    "ı": "i", "İ": "i", "I": "i", "ß": "ss", "đ": "d", "Đ": "d",
    "ø": "o", "Ø": "o", "æ": "ae", "Æ": "ae",
})


def normalize(text: str) -> str:
    """Fold case, diacritics and separators; keep Greek and Cyrillic letters.

    NFKD decomposition strips Latin accents and Greek tonos alike (a -> a), which is what
    we want: reports are inconsistent about accents. It also maps the MICRO SIGN U+00B5
    to a real mu, which matters because most Greek reports here use the wrong codepoint.
    """
    if not isinstance(text, str):
        return ""
    text = text.translate(_PRE).lower()
    text = unicodedata.normalize("NFKD", text)
    text = "".join(ch for ch in text if not unicodedata.combining(ch))
    text = text.replace("­", "")               # soft hyphen
    text = re.sub(r"[_\-/\\]+", " ", text)
    text = re.sub(r"[ \t]+", " ", text)
    return text


_SENT_SPLIT = re.compile(r"(?<=[.;!?])\s+|\n+")


def clauses(text: str):
    """Split into clauses, then attach `header:` lines to the value that follows.

    A report line reading `Fractures :` followed by `Aucune.` is one statement. Splitting
    on punctuation alone separates the anatomy from its negation and flips the label.
    """
    norm = normalize(text)
    raw = [c.strip() for c in _SENT_SPLIT.split(norm) if c and c.strip()]

    merged = []
    for i, c in enumerate(raw):
        # A fragment ending in a colon is a heading for the next fragment. Structured
        # English reports write long ones - "lateral compartment (meniscus, collateral
        # ligament complex, cartilage):" is eight words - so the cap is generous.
        #
        # The heading is emitted ONLY joined to its value, never also on its own. A bare
        # `Fractures :` contains the anatomy and no polarity cue, so scoring it alone
        # reads it as an assertion - and the very next line is `Aucune.` Structured
        # reports are built out of exactly this shape, so keeping the bare heading turns
        # every negated section header into a false positive. The joined clause is a
        # superset of the heading's text, so nothing is lost by dropping it.
        if c.endswith(":") and len(c.split()) <= 14 and i + 1 < len(raw):
            merged.append(c + " " + raw[i + 1])
            continue
        merged.append(c)
    # Comma-separated enumerations inside a long clause hide separate assertions.
    out = []
    for c in merged:
        out.append(c)
        if len(c.split()) > 25:
            out.extend(p.strip() for p in c.split(",") if len(p.split()) > 2)
    return out


def _rx(*alts: str) -> re.Pattern:
    return re.compile("|".join(alts))


NEGATION = _rx(
    # `none` and `nil` matter more than their frequency suggests: a structured report
    # writes the anatomy as a heading and the finding as a one-word value beneath it,
    # so `FRACTURE:` / `None.` is the entire statement and missing that one word flips
    # the label on every section that was checked and found clear.
    r"\bno\b", r"\bnot\b", r"\bnone\b", r"\bnil\b", r"\babsent\b",
    r"\bwithout\b", r"\bnegative\b", r"\babsence\b",
    r"\bno evidence\b", r"\bunremarkable\b", r"\bfree of\b",
    r"\bsin\b", r"\bno hay\b", r"\bausencia\b", r"\bausentes?\b", r"\bningun[ao]?\b",
    r"\bpas de\b", r"\bsans\b", r"\baucune?\b", r"\brien\b",
    r"\bgeen\b", r"\bzonder\b", r"\bniet\b",
    r"\bkeine?\b", r"\bohne\b", r"\bnicht\b",
    r"\byok\b", r"\byoktur\b", r"izlenmemekte", r"saptanmadi", r"\bdegil\b",
    r"gozlenmemekte", r"mevcut degil", r"eslik etmiyor", r"\bizlenmedi\b",
    r"\bnema\b", r"\bbez\b", r"\bnisu\b", r"\bnije\b",
    r"\bδεν\b", r"\bχωρις\b", r"ουδεν",
    r"\bбез\b", r"\bне\b", r"липсва", r"\bняма\b",
)

NORMALITY = _rx(
    r"\bnormal", r"\bintact\b", r"\bpreserved\b", r"\bwithin normal limits\b",
    r"limites normales", r"\bconservad", r"\bintegr", r"\bnormales\b",
    r"\bdoga(l|ll)\b", r"korunmus", r"\bnormaldir\b", r"olagan",
    r"\buredn", r"\bocuvan", r"\bodrzan", r"\bintakt",
    r"φυσιολογικ", r"ακεραι",
    r"unauffallig", r"regelrecht",
    r"нормал", r"запазен", r"съхранен", r"\bбез особености\b",
    r"\bgaaf\b", r"\bnormaal\b",
)

UNCERTAIN = _rx(
    r"\bpossible\b", r"\bprobable\b", r"\bsuspicious\b", r"\bsuspected\b",
    r"cannot (be )?exclude", r"\bmay\b", r"\bquestionable\b", r"\bequivocal\b",
    r"\bposible\b", r"sin criterios categoricos", r"\bdudos",
    r"\bmuhtemel\b", r"\bolasi\b", r"\bsupheli\b", r"\bizlenim",
    r"\bmoguce\b", r"\bvjerojatno\b", r"\bsumnja\b",
    r"πιθαν", r"υποπτ",
    r"\bmoglich", r"\bverdachtig", r"\bfraglich", r"\bv\.a\.\b",
    r"\bвъзможно\b", r"\bвероятно\b", r"суспект",
    r"\bmogelijk\b", r"\bverdacht\b",
)

TEAR = _rx(
    # `disrupt`, not `\bdisruption\b`: "the ACL is disrupted" is the commonest English
    # phrasing for a complete tear and the noun form misses it entirely.
    r"\btear", r"\btorn\b", r"\brupture", r"disrupt", r"discontinuit",
    r"\bavuls", r"\blacerat",
    r"\brotura\b", r"\broturas\b", r"\bruptura", r"\bdesgarro", r"\broto\b",
    r"\bdechirure", r"\bdechire",
    r"\bscheur", r"\bruptuur", r"gescheurd",
    # German compounds the pathology onto the anatomy - Innenmeniskusriss,
    # Kreuzbandruptur, Meniskusabriss - so these stems must not carry a leading word
    # boundary or the entire German subcorpus reads as negative.
    r"riss\b", r"einriss", r"ruptur", r"zerreiss", r"\blasion",
    r"kontinuitatsunterbrechung",
    r"\byirtik", r"\byirtig", r"\bkopma\b", r"butunluk kaybi",
    r"\bpuknuce", r"\bprekid\b", r"\bpukotin",
    r"ρηξη", r"ρηξις", r"ρηγμα",
    r"руптура", r"разкъсв", r"разрив", r"скъсв",
)

DEGEN = _rx(
    r"degenerat", r"\bmucoid\b", r"\bmyxoid\b", r"\bfray", r"\bfissur",
    r"dejeneratif", r"\bmukoid\b", r"degenerativn", r"εκφυλιστ", r"дегенерат",
    r"\bmuco ?ide\b", r"aufgefasert",
)

INJURY = _rx(
    r"\binjur", r"\bsprain", r"\blesion", r"\blasion", r"\bedema\b", r"\boedema\b",
    r"\bodem\b", r"\bedem\b", r"\bοιδημα", r"\bодем", r"\bедем", r"\bstrain\b",
    r"\bhigh signal\b", r"\bsignal alteration\b", r"\bhiperintens", r"\bhyperintens",
    r"\bthicken", r"\bzadebljanje\b", r"\bverdikking\b", r"\bdistenzij",
    r"\blaksite\b", r"\blaxity\b", r"\bpartial\b", r"\bparcijaln", r"\bparcial",
    r"\bpartiel", r"\bpartiell",
)

ANAT = {
    "ACL": _rx(
        r"anterior cruciate", r"\bacl\b",
        r"cruzado anterior", r"\blca\b",
        r"croise anterieur",
        r"voorste kruisband", r"\bvkb\b",
        r"vorderes kreuzband", r"vorderen kreuzband", r"vordere kreuzband",
        r"on capraz", r"\bocb\b",
        r"prednji krizni", r"prednjeg krizn",
        r"προσθι[οα][^ ]* χιαστ", r"προσθιου χιαστου", r"χιαστο[^ ]* συνδεσμ",
        r"предна кръстна", r"предната кръстна",
        # Plural, unqualified: reports routinely clear both cruciates in one clause
        # ("Ligamentos cruzados y colaterales dentro de limites normales").
        r"cruciate ligaments", r"ligamentos cruzados", r"ligaments croises",
        r"kruisbanden", r"kreuzbander", r"capraz baglar", r"krizn[a-z]* ligament[a-z]*",
        r"χιαστοι συνδεσμ", r"χιαστων συνδεσμ", r"кръстните връзки", r"кръстни връзки",
    ),
    "MCL": _rx(
        r"medial collateral", r"\bmcl\b", r"tibial collateral",
        r"colateral medial", r"colateral interno", r"\blcm\b",
        r"collateral medial", r"collateral interne",
        r"mediale collaterale", r"binnenband",
        r"innenband", r"mediales? kollateral",
        r"\bic yan bag", r"medial kollateral", r"\biyb\b",
        r"medijalni kolateraln", r"medijalnog kolateraln",
        r"εσω πλαγι", r"εσωτερικο πλαγι",
        r"медиален колатерал", r"вътрешна странична",
        # "Ligamentos cruzados y colaterales" separates the noun from its adjective, so
        # the adjective has to stand alone as a cue.
        r"\bcolaterales\b", r"\bcollateraux\b", r"\bcollateralen\b", r"\bkolateralni\b",
        r"collateral ligaments", r"ligamentos colaterales", r"ligaments collateraux",
        r"collaterale banden", r"kollateralbander", r"seitenbander", r"yan baglar",
        r"kolateraln[a-z]* ligament[a-z]*", r"πλαγιοι συνδεσμ", r"πλαγιων συνδεσμ",
        r"колатерални връзки", r"страничните връзки",
    ),
    "Medial Meniscus": _rx(
        r"medial meniscus", r"\bmm\b(?= tear)", r"medial menisc",
        r"menisco medial", r"menisco interno",
        r"menisque medial", r"menisque interne",
        r"mediale meniscus", r"binnenmeniscus",
        r"innenmeniskus", r"medialen? meniskus", r"innenmeniskushinterhorn",
        r"medyal menisk", r"\bic menisk",
        r"medijalni meniskus", r"medijalnog meniskusa", r"medijalnom meniskusu",
        r"εσω μηνισκ", r"μηνισκ[^ ]* του εσω", r"εσω διαμερισμα[^.]{0,40}μηνισκ",
        r"медиалния менискус", r"медиален менискус", r"вътрешния менискус",
    ),
    "Lateral Meniscus": _rx(
        r"lateral meniscus", r"lateral menisc",
        r"menisco lateral", r"menisco externo",
        r"menisque lateral", r"menisque externe",
        r"laterale meniscus", r"buitenmeniscus",
        r"aussenmeniskus", r"lateralen? meniskus",
        r"lateral menisk", r"\bdis menisk",
        r"lateralni meniskus", r"lateralnog meniskusa", r"lateralnom meniskusu",
        r"εξω μηνισκ", r"μηνισκ[^ ]* του εξω", r"εξω διαμερισμα[^.]{0,40}μηνισκ",
        r"латералния менискус", r"латерален менискус", r"външния менискус",
    ),
}

# Osteoarthritis is rarely written as "osteoarthritis". It is written as cartilage loss,
# chondropathy grade, joint space narrowing, or osteophytes - scoped to a compartment.
OA_EVIDENCE = _rx(
    r"osteoarthrit", r"\barthros", r"\bgonarthros", r"\bosteoarthros",
    r"chondropath", r"chondromalac", r"condropat", r"condromalac",
    r"cartilage loss", r"cartilage thinning", r"chondral (loss|defect|ulcer|thinning)",
    r"osteophyt", r"osteofit", r"osteofyt", r"osteofito", r"osteophyten",
    r"joint space narrowing", r"pinzamiento articular",
    r"kikirdak kayb", r"kikirdak incelme", r"kondropati", r"kondral",
    r"kraakbeen(lijden|verlies)", r"gonartrose", r"artrose",
    r"knorpel(verlust|schaden|defekt)", r"arthrose", r"gonarthrose",
    r"hrskavic", r"hondromalac", r"artroz", r"osteoartrit",
    r"χονδρ[^ ]*παθ", r"αρθριτ", r"αρθρωσ", r"οστεοφυτ",
    r"αρθρικου χονδρου", r"εξαλειψη του αρθρικου χονδρου",
    r"артроз", r"хондропат", r"остеофит", r"хрущял[^.]{0,30}(изтън|увред|дефект)",
    r"ulcera[s]? condral", r"cartilago[^.]{0,25}(perdida|adelgaz)",
    r"icrs grade", r"outerbridge",
)

COMPARTMENT = {
    "Medial OA": _rx(
        r"medial (femorotibial|tibiofemoral|compartment)",
        r"compartimento femorotibial medial", r"femorotibial interno",
        r"mediaal femorotibiaal", r"mediale femorotibial",
        r"medial femorotibial", r"medialen kompartiment", r"innere[sn]? kompartiment",
        r"medyal femorotibial", r"ic kompartman", r"medyal kompartman",
        r"medijaln[^ ]* (femorotibi|odjelj|kompartm)",
        r"εσω διαμερισμα", r"εσω κνημιαι", r"εσω μηριαι",
        r"медиалн[^ ]* (компартм|отдел|тибиал|феморотиб)",
        r"medial (femoral|tibial) (condyle|plateau)", r"condilo femoral medial",
        r"medialen? (femurkondyl|tibiaplateau)", r"mediale femorale condyl",
    ),
    "Lateral OA": _rx(
        r"lateral (femorotibial|tibiofemoral|compartment)",
        r"compartimento femorotibial lateral", r"femorotibial externo",
        r"lateraal femorotibiaal", r"laterale femorotibial",
        r"lateral femorotibial", r"lateralen kompartiment", r"aussere[sn]? kompartiment",
        r"dis kompartman", r"lateral kompartman",
        r"lateraln[^ ]* (femorotibi|odjelj|kompartm)",
        r"εξω διαμερισμα", r"εξω κνημιαι", r"εξω μηριαι",
        r"латералн[^ ]* (компартм|отдел|тибиал|феморотиб)",
        r"lateral (femoral|tibial) (condyle|plateau)", r"condilo femoral lateral",
        r"lateralen? (femurkondyl|tibiaplateau)", r"laterale femorale condyl",
    ),
    "PF OA": _rx(
        r"patellofemoral", r"femoropatellar", r"femoropatelar", r"patelofemoral",
        r"retropatellar", r"retrorotulian", r"\btrochlea", r"\btroclea", r"\btroklea",
        r"\bpatella\b", r"\bpatellar\b", r"\brotulian", r"\brotula\b", r"\bpatele\b",
        r"\bpatellae?\b", r"patellofemoraal", r"femoropatellair",
        r"επιγονατιδ", r"μηροεπιγονατιδ", r"τροχιλ",
        r"пател", r"феморопател", r"тролх",
        r"anterior compartment", r"compartimento anterior", r"prednj[^ ]* odjeljk",
    ),
}

# Self-declaring findings: the term itself is the finding.
DIRECT = {
    "Effusion": _rx(
        r"\beffusion", r"joint fluid", r"intra ?articular fluid", r"\bhydrops\b",
        r"derrame articular", r"\bderrame\b", r"liquido articular",
        r"epanchement",
        r"gewrichtsvocht", r"\bvocht\b", r"gewrichtseffusie",
        r"gelenkerguss", r"\berguss\b", r"gelenksergu",
        # "diz eklemi ici sivi miktari ... artmis" - the noun takes a possessive suffix,
        # so `eklem ` alone misses. Match the stem plus any suffix.
        r"eklem\w* ic\w* sivi", r"efuzyon", r"eklem sivisi",
        r"sivi (miktari|artisi|birikimi)", r"sivi artis", r"\bsivi\b[^.]{0,25}artmis",
        r"\bizljev", r"\bizliv", r"zglobn[^ ]* tekucin", r"\bhidrops\b",
        r"αρθρικ[^ ]* υγρ", r"υγρου ενδαρθρικα", r"ενδαρθρικ[^ ]* υγρ", r"ποσοτητα υγρου",
        r"ενδαρθρικ", r"αρθρικη συλλογη", r"υγρο στην αρθρωση", r"υγρου στην αρθρωση",
        r"ставен излив", r"излив", r"ставна течност", r"синовиална течност",
    ),
    "Synovitis": _rx(
        r"synovit", r"sinovit", r"synovial (thickening|proliferation|hypertroph)",
        r"synoviale? (verdikking|proliferatie)",
        r"synovialitis", r"synovialis(verdickung|proliferation)",
        r"sinovijalitis", r"zadebljanje sinovij",
        r"υμενιτιδα", r"συνοβιτιδα", r"υμενικ[^ ]* υπερτροφ", r"αρθρικου υμεν",
        r"синовит", r"синовиал[^ ]* (задебел|пролифер)",
        r"verdikkingen van (het )?synovium", r"pannus",
    ),
    "Baker's": _rx(
        r"baker", r"popliteal cyst", r"quiste popliteo", r"quistes popliteos",
        r"kyste poplite", r"popliteale? cyst", r"poplitealzyste", r"bakerzyste",
        r"popliteal kist", r"\bbakerova\b", r"poplitealn[^ ]* cist",
        r"κυστη baker", r"πολυχωρη συνοβιακη κυστη", r"κυστη του baker",
        r"киста на бейкър", r"бейкърова киста", r"поплитеална киста",
        r"gastrocnemio ?semimembranos", r"gastrocnemius semimembranosus burs",
    ),
    "Contusion": _rx(
        r"\bcontusion", r"bone bruise", r"bone marrow (o?edema|contusion)",
        r"\bkontuz", r"medular bone o?edema", r"marrow o?edema",
        r"edema oseo", r"edema de medula osea",
        r"oedeme osseux",
        r"botcontusie", r"botoedeem", r"beenmergoedeem", r"botmergoedeem",
        r"knochenmarkodem", r"knochenodem", r"kontusion",
        r"kemik kontuzyonu", r"kemik iligi odemi", r"kemik odemi",
        r"kostani edem", r"edem kosti", r"kontuzij",
        r"οστεομυελικ[^ ]* οιδημα", r"οστικο οιδημα", r"μυελικο οιδημα",
        r"костномозъчен едем", r"костен едем", r"контузионен",
    ),
    "Fracture": _rx(
        r"\bfractur", r"\bfract\b",
        r"\bfractura", r"\bfracturas\b",
        r"\bfractuur", r"\bbreuk\b",
        r"\bfraktur", r"\bbruch\b",
        r"\bkirik\b", r"\bkirigi\b",
        r"\bprijelom", r"impresijsk[^ ]* fraktur",
        r"καταγμα", r"καταγματ",
        r"фрактур", r"счупван", r"фисур",
        r"insufficiency fracture", r"stress fracture", r"avulsion fracture",
        r"subchondral fracture", r"subkondral kiri",
    ),
}

# Terms that look like a finding but are not the finding being scored.
DECOY = {
    "Fracture": _rx(r"no fracture", r"microfractur", r"\bfracture (risk|prophyla)"),
    "Baker's": _rx(r"meniscal cyst", r"quiste meniscal", r"ganglion"),
}

PAIRED = {"ACL", "MCL", "Medial Meniscus", "Lateral Meniscus"}
OA_TARGETS = {"Medial OA", "Lateral OA", "PF OA"}

STEM_MENISCUS = _rx(r"menisc\w*", r"menisk\w*", r"μηνισκ\w*", r"мениск\w*")
STEM_CRUCIATE = _rx(r"cruciate", r"cruzado", r"croise", r"kruisband", r"kreuzband",
                    r"capraz bag\w*", r"krizn\w*", r"χιαστ\w*", r"кръстн\w*",
                    r"\bacl\b", r"\bpcl\b", r"\blca\b", r"\blcp\b", r"\bvkb\b",
                    r"\bhkb\b", r"\bocb\b", r"\bacb\b")
STEM_COLLATERAL = _rx(r"collateral\w*", r"colateral\w*", r"kollateral\w*",
                      r"collaterale\w*", r"kolateraln\w*", r"yan bag\w*",
                      r"πλαγι\w*", r"колатерал\w*", r"странич\w*",
                      r"innenband\w*", r"aussenband\w*", r"binnenband\w*",
                      r"\bmcl\b", r"\blcl\b", r"\blcm\b", r"\biyb\b")

SIDE_MEDIAL = _rx(r"\bmedial\w*", r"\bmedyal\w*", r"\bmedijaln\w*", r"\bmediaal\w*",
                  r"\bmediale\w*", r"\binterno\w*", r"\binterne\w*", r"\binnen\w*",
                  r"\bic\b", r"\bunutarnj\w*", r"\bεσω\w*", r"\bεσωτερικ\w*",
                  r"\bмедиал\w*", r"\bвътреш\w*", r"\btibial collateral\b")
SIDE_LATERAL = _rx(r"\blateral\w*", r"\bexterno\w*", r"\bexterne\w*", r"\bdis\b",
                   r"\blateraln\w*", r"\baussen\w*", r"\bbuiten\w*", r"\bεξω\w*",
                   r"\bεξωτερικ\w*", r"\bлатерал\w*", r"\bвъншн\w*",
                   r"\bfibular collateral\b", r"\bvanjsk\w*")
SIDE_ANTERIOR = _rx(r"\banterior\w*", r"\bant\b", r"\bon\b", r"\bprednj\w*",
                    r"\bvorder\w*", r"\bvoorste\b", r"\bπροσθι\w*", r"\bпредн\w*",
                    r"\banteriyor\w*", r"\bavant\b", r"\banterieur\w*")

# Fracture is the target whose stem varies most across the corpus.
STEM_FRACTURE = _rx(r"fractur\w*", r"fraktur\w*", r"fractuur\w*", r"\bfract\b",
                    r"kiri[kg]\w*", r"prijelom\w*", r"lom kosti", r"\bbreuk\w*",
                    r"\bbruch\w*", r"καταγμα\w*", r"καταγματ\w*", r"фрактур\w*",
                    # NOT a bare `fissur\w*`: "fisuras condrales" and "full thickness
                    # fissures in the articular cartilage" describe cartilage, not bone.
                    r"счупван\w*", r"fisur\w* (osea|oseas|kost)", r"fissur\w* kost")

STEM_OA_COMPARTMENT = _rx(r"compartment\w*", r"compartimento\w*", r"compartiment\w*",
                          r"kompartman\w*", r"kompartiment\w*", r"odjelj\w*",
                          r"διαμερισμα\w*", r"компартм\w*", r"\bотдел\w*",
                          r"femorotibial\w*", r"femorotibiaal\w*", r"tibiofemoral\w*",
                          r"femoro tibial\w*", r"κνημιαι\w*", r"μηριαι\w*",
                          r"femoral condyl\w*", r"tibial plateau\w*",
                          r"condilo femoral", r"platillo tibial", r"tibiaplateau\w*",
                          r"femurkondyl\w*", r"femoralne? kondil\w*",
                          r"tibijaln\w* plato", r"femoral kondil\w*",
                          r"tibia plato", r"tibyal plato")


def _near(clause: str, stem_rx: re.Pattern, qual_rx: re.Pattern, window: int = 55):
    """True if a stem match has a qualifier within `window` characters either side.

    Character windows rather than token windows, because word order differs: English
    puts the side before the noun, Greek and Bulgarian often after, and Turkish
    attaches it as a separate preceding adjective.
    """
    for m in stem_rx.finditer(clause):
        lo = max(0, m.start() - window)
        hi = min(len(clause), m.end() + window)
        if qual_rx.search(clause[lo:hi]):
            return True
    return False


STEM_RULES = {
    "ACL": (STEM_CRUCIATE, SIDE_ANTERIOR),
    "MCL": (STEM_COLLATERAL, SIDE_MEDIAL),
    "Medial Meniscus": (STEM_MENISCUS, SIDE_MEDIAL),
    "Lateral Meniscus": (STEM_MENISCUS, SIDE_LATERAL),
    "Medial OA": (STEM_OA_COMPARTMENT, SIDE_MEDIAL),
    "Lateral OA": (STEM_OA_COMPARTMENT, SIDE_LATERAL),
}

SEV_LOW = _rx(
    r"\bsmall\b", r"\bminimal\b", r"\btrace\b", r"\bmild\b", r"\bslight\b",
    r"\btiny\b", r"\bscant\b", r"\bmimimal\b", r"\bdiscrete\b", r"\bfocal\b",
    r"\bleve\b", r"\bminim", r"\bpeque", r"\bligero\b", r"\bescaso\b", r"\bdiscreto\b",
    r"\bhafif\b", r"\baz miktarda\b", r"\bsilik\b",
    r"\bmanja\b", r"\bmanji\b", r"\bblago\b", r"\bdiskretn", r"\bmalo\b",
    r"\bgering", r"\bdiskret", r"\bkleine?r?\b", r"\bwenig\b", r"\bzarte?\b",
    r"\bbeperkte?\b", r"\bgeringe\b", r"\bweinig\b", r"\blichte?\b",
    r"\bηπι", r"\bμικρ", r"\bελαχιστ",
    r"\bминимал", r"\bлек", r"\bмалк", r"\bнеголям",
)

SEV_HIGH = _rx(
    r"\blarge\b", r"\bmarked\b", r"\bmassive\b", r"\bsevere\b", r"\bextensive\b",
    r"\bmoderate\b", r"\bgross\b", r"\bsignificant\b", r"\babundant\b", r"\btense\b",
    r"\bmoderad", r"\bimportante\b", r"\bsevera?\b", r"\bmarcad", r"\bcuantios",
    r"\bbelirgin\b", r"\byaygin\b", r"\bileri\b", r"\bciddi\b", r"\bbol\b",
    r"\bopsezan\b", r"\bveliki\b", r"\bizrazit", r"\bznacajn", r"\bumjeren",
    r"\bausgepragt", r"\bdeutlich", r"\bmassiv", r"\bmassig", r"\bgross",
    r"\buitgebreid", r"\bgevorderd", r"\bveel\b", r"\bmatige?\b",
    r"\bμετρι", r"\bμεγαλ", r"\bεκτεταμεν", r"\bευμεγεθ", r"\bσοβαρ",
    r"\bголям", r"\bизразен", r"\bзначим", r"\bумерен", r"\bобилен",
)

# OA is often asserted for the whole joint rather than per compartment
# ("tricompartmental osteoarthritis", "gonarthrose"). Those statements are evidence for
# all three OA targets.
GLOBAL_OA = _rx(
    r"tri ?compartment", r"all three compartment", r"global(ised)? (oa|osteoarthrit)",
    r"\bgonarthros", r"\bgonartros", r"\bgonarthrose", r"\bgonartrose",
    r"osteoarthritis of the knee", r"artrosis (de |)(la )?rodilla", r"knee osteoarthrit",
    r"\bdiz osteoartrit", r"\bgonartroz", r"artroza koljena",
    r"οστεοαρθριτιδα", r"αρθριτιδα του γονατος",
    r"артроза на колянната", r"гонартроз",
    r"degenerative joint disease", r"\bdjd\b",
)

# A bare "bone marrow oedema" is not a contusion when it sits under a cartilage defect:
# subchondral oedema beneath a worn compartment is reactive degenerative signal, and
# reading it as a bruise turns every osteoarthritic knee into a trauma case.
DEGENERATIVE_MARROW = _rx(
    r"subchondral", r"subcondral", r"subkondral", r"supkondraln", r"subchondraln",
    r"υποχονδρι", r"субхондрал",
    r"\bcyst", r"\bquist", r"\bzyste\b", r"\bcistic", r"reactive", r"reactivo",
)

TRAUMA = _rx(
    r"\bbruise\b", r"\bcontusion", r"\bkontuz",
    r"\btrauma", r"\bimpaction\b", r"\bpivot shift\b", r"\bkissing\b",
    r"\bacute\b", r"\bagudo\b", r"\bakut", r"\bpivot kaymasi\b",
    r"\bbone bruise\b", r"\bbotcontusie\b",
    r"\bконтузион", r"\bμωλωπ", r"\bkontuzij",
)


def _polarity(clause: str) -> str:
    """Classify one clause as positive, negative or uncertain for a matched term.

    Scope is the whole clause. Clause segmentation already keeps statements short, and
    a window in characters mis-scopes badly across languages with different word orders -
    Turkish puts its negator at the end of the sentence, English at the front.
    """
    if UNCERTAIN.search(clause):
        return "uncertain"
    if NEGATION.search(clause):
        return "negative"
    if NORMALITY.search(clause):
        # "meniscus normal" negates; "normal ... but tear" does not.
        if TEAR.search(clause) or re.search(r"\bgrade [34]\b", clause):
            return "positive"
        return "negative"
    return "positive"


class _Matcher:
    """Phrase lexicon first, stem+side proximity as the fallback.

    Exposes `.search` so it drops into the same slot as a compiled pattern.
    """

    def __init__(self, phrase_rx, stem=None, side=None, window=55):
        self.phrase_rx = phrase_rx
        self.stem = stem
        self.side = side
        self.window = window

    def search(self, clause):
        m = self.phrase_rx.search(clause)
        if m is not None:
            return m
        if self.stem is not None and _near(clause, self.stem, self.side, self.window):
            return self.stem.search(clause)
        return None


ANAT_MATCH = {tgt: _Matcher(ANAT[tgt], *STEM_RULES[tgt]) for tgt in PAIRED}
COMPARTMENT_MATCH = {
    "Medial OA": _Matcher(COMPARTMENT["Medial OA"], *STEM_RULES["Medial OA"]),
    "Lateral OA": _Matcher(COMPARTMENT["Lateral OA"], *STEM_RULES["Lateral OA"]),
    "PF OA": _Matcher(COMPARTMENT["PF OA"]),
}
DIRECT_MATCH = {
    tgt: _Matcher(_rx(rx.pattern, STEM_FRACTURE.pattern) if tgt == "Fracture" else rx)
    for tgt, rx in DIRECT.items()
}


def _severity(clause: str) -> float:
    """Weight one positive mention by how emphatic the sentence is.

    Ordered, not calibrated. A "moderate effusion" must outrank a "trace effusion" and
    both must outrank silence; the absolute numbers do not matter to AUC.
    """
    high = SEV_HIGH.search(clause) is not None
    low = SEV_LOW.search(clause) is not None
    if high and not low:
        return 1.0
    if low and not high:
        return 0.45
    return 0.75                       # unqualified mention


def _score_clauses(cls, anat_rx, path_rx=None, decoy_rx=None, context_penalty=None,
                   context_bonus=None):
    """Accumulate graded evidence over clauses for one target.

    Returns (score, confidence, n_pos, n_neg). Positives are graded by severity and by
    optional context regexes; negatives only matter when nothing positive was found,
    because reports assert normality for every structure they check.
    """
    n_pos = n_neg = n_unc = 0
    best = 0.0
    for c in cls:
        m = anat_rx.search(c)
        if not m:
            continue
        if decoy_rx is not None and decoy_rx.search(c):
            continue
        if path_rx is not None and not path_rx.search(c):
            if NORMALITY.search(c) and not NEGATION.search(c):
                n_neg += 1
            continue
        pol = _polarity(c)
        if pol == "positive":
            n_pos += 1
            w = _severity(c)
            if context_penalty is not None and context_penalty.search(c):
                w *= 0.45
            if context_bonus is not None and context_bonus.search(c):
                w = min(1.0, w * 1.35)
            best = max(best, w)
        elif pol == "negative":
            n_neg += 1
        else:
            n_unc += 1
            best = max(best, 0.30)

    if n_pos or n_unc:
        # 0.52 .. 0.95, ordered by the strongest single mention, nudged by repetition.
        score = min(0.95, 0.50 + 0.42 * best + 0.03 * min(n_pos, 3))
        conf = min(1.0, 0.55 + 0.15 * n_pos)
    elif n_neg:
        score = max(0.04, 0.20 - 0.04 * n_neg)
        conf = min(0.9, 0.45 + 0.12 * n_neg)
    else:
        score, conf = 0.28, 0.05          # silence sits above asserted-negative
    return score, conf, n_pos, n_neg


def extract(report: str) -> dict:
    """Extract twelve (score, confidence) pairs from one report."""
    cls = clauses(report)
    out = {}
    path_paired = _rx(TEAR.pattern, DEGEN.pattern, INJURY.pattern)

    for tgt in TARGETS:
        if tgt in PAIRED:
            s, c, npos, nneg = _score_clauses(cls, ANAT_MATCH[tgt], path_paired)
        elif tgt in OA_TARGETS:
            s, c, npos, nneg = _score_clauses(cls, COMPARTMENT_MATCH[tgt], OA_EVIDENCE)
        elif tgt == "Contusion":
            # Reactive subchondral oedema under a cartilage defect is osteoarthritis,
            # not a bruise. Explicit trauma wording pushes the other way.
            s, c, npos, nneg = _score_clauses(cls, DIRECT_MATCH[tgt], None, DECOY.get(tgt),
                                              context_penalty=DEGENERATIVE_MARROW,
                                              context_bonus=TRAUMA)
        else:
            s, c, npos, nneg = _score_clauses(cls, DIRECT_MATCH[tgt], None, DECOY.get(tgt))
        out[tgt] = s
        out[tgt + "__conf"] = c
        out[tgt + "__npos"] = npos
        out[tgt + "__nneg"] = nneg

    # --- cross-target corrections ------------------------------------------ #
    # A whole-joint osteoarthritis statement is evidence for every compartment that was
    # not separately assessed. Without this, "incipient OA of all three compartments"
    # scores zero on all three OA targets.
    g_hits = [c for c in cls if GLOBAL_OA.search(c) and _polarity(c) == "positive"]
    if g_hits:
        gscore = 0.50 + 0.42 * max(_severity(c) for c in g_hits)
        for tgt in OA_TARGETS:
            if out[tgt + "__npos"] == 0 and out[tgt + "__nneg"] == 0:
                out[tgt] = max(out[tgt], gscore * 0.92)
                out[tgt + "__conf"] = max(out[tgt + "__conf"], 0.4)

    # Synovitis is frequently visible on the images and absent from the text, so silence
    # is weak evidence of absence here in a way it is not for other findings. Effusion is
    # its most reliable textual proxy - the two share a mechanism - so a silent synovitis
    # inherits a fraction of the effusion evidence instead of falling to the floor.
    if out["Synovitis__npos"] == 0 and out["Synovitis__nneg"] == 0:
        out["Synovitis"] = max(out["Synovitis"], 0.28 + 0.45 * (out["Effusion"] - 0.28))

    return out

### The lexicon hole, and what to do about it

The dangerous failure of a rule extractor is silent: a rule that never fires does not
raise, it emits a negative. A lexicon thick in English and thin in Greek does not look
broken — it looks like a corpus where Greek patients have fewer findings. And because
language tracks the reporting institution, which tracks the scanner and the population,
that gap is a bias aligned with a site rather than noise that averages out.

The fix here is to fit a character n-gram model **to the rule scores**, over all 4,407
reports, out of fold. `char_wb` 3–5 needs no tokeniser, stemmer or stopword list, so
Greek and Turkish are handled on the same footing as English; the model picks up the
phrasings that co-occur with rule-positive reports and scores them without anyone having
written them into the lexicon.

The precise claim matters, because the stronger one is false. The model cannot find
findings in a language the lexicon misses *entirely* — its only supervision is the rule
output, so with no variation there is nothing to learn. What it repairs is **partial**
coverage, which is the real situation. `tests/test_distillation.py` measures exactly
this: on a corpus where 64% of positive reports use phrasings outside the lexicon, the
rules sit at chance (0.500) on those reports and the blend reaches 0.940.

Each target is gated on how well the text model recovered the rules out of fold. Below
a correlation of 0.15 it contributes nothing rather than noise — the negative control in
that test file confirms the gate fires when coverage is zero.

The other trap is on the way back. Combining as ranks is right, but reading the combined
rank back through the *sorted* rule scores re-quantises everything into the rules' four
atoms, and since most mass sits in one enormous "silent" atom, that throws away precisely
the ordering the text model was added to supply. Interpolating between the distinct
levels instead keeps the prevalence and the meaning of 0.5 while preserving the ordering.

In [4]:
#
# This is the part the source notebooks do not have, and it exists to patch the one
# failure the rule extractor cannot see from inside itself.
#
# `rsna-knee-baseline-v1` names the problem precisely: a rule that never fires emits a
# negative rather than an error, so a lexicon thin in one language looks like a
# population with fewer findings, and language tracks site. `rsna-knee-eda-to-2-5d`
# fits a TF-IDF teacher, but on the 58 annotated studies only - far too few rows for a
# 190k-feature model, which is why its reported OOF AUC sits near chance on several
# targets.
#
# Fitting the text model to the *rule scores* instead uses all 4,407 reports. The model
# then learns which character n-grams co-occur with rule-positive reports, in every
# language present, and can score a Greek report the lexicon was silent on. Out-of-fold
# and grouped on report text so it cannot memorise its own training signal.

def distill(reports, rule_scores, groups, n_splits=5, seed=2026, alpha=1.0,
            verbose=True):
    """Fit a char+word TF-IDF ridge to the rule scores, out-of-fold.

    reports      : list[str], length N
    rule_scores  : (N, 12) float, the graded rule output
    groups       : (N,) int, fold assignment (hash of report text - duplicates together)
    returns      : (N, 12) float, OOF text predictions on the same 0-1 scale

    Ridge on the graded score rather than logistic on a binarisation, because §1 of the
    source notebook is right that only order is read and grading carries more signal
    than a threshold does.
    """
    from sklearn.feature_extraction.text import TfidfVectorizer
    from sklearn.linear_model import Ridge
    from scipy import sparse

    texts = [normalize(r) for r in reports]
    rule_scores = np.asarray(rule_scores, dtype=np.float32)
    groups = np.asarray(groups)

    # char_wb 3-5 is what makes this multilingual: it needs no tokeniser, no stemmer and
    # no stopword list, so Greek and Turkish are handled on the same footing as English.
    char = TfidfVectorizer(analyzer="char_wb", ngram_range=(3, 5), min_df=3,
                           max_features=200_000, sublinear_tf=True)
    word = TfidfVectorizer(ngram_range=(1, 2), min_df=2, max_features=80_000,
                           sublinear_tf=True)
    X = sparse.hstack([char.fit_transform(texts), word.fit_transform(texts)],
                      format="csr")

    oof = np.zeros_like(rule_scores)
    folds = sorted(set(groups.tolist()))
    for f in folds:
        va = np.flatnonzero(groups == f)
        tr = np.flatnonzero(groups != f)
        if len(tr) < 20 or len(va) == 0:
            oof[va] = rule_scores[va]
            continue
        m = Ridge(alpha=alpha, solver="sparse_cg")
        m.fit(X[tr], rule_scores[tr])
        oof[va] = m.predict(X[va])

    oof = np.clip(oof, 0.02, 0.98)

    # How well the text model reproduces the rules out of fold, per target. This is the
    # gate, not a diagnostic: the text model earns its weight by disagreeing with the
    # rules where the rules are silent, but a model that cannot even recover them
    # out-of-fold has learned nothing and must not be blended in. Targets whose lexicon
    # is thin, or whose positives are too rare for the ridge to find, land here.
    corr = np.zeros(rule_scores.shape[1], np.float32)
    for j in range(rule_scores.shape[1]):
        if np.std(oof[:, j]) > 1e-8 and np.std(rule_scores[:, j]) > 1e-8:
            corr[j] = float(np.corrcoef(oof[:, j], rule_scores[:, j])[0, 1])
    if verbose:
        print("distilled text model, OOF corr with rules per target:")
        for t, a in zip(TARGETS, corr):
            print(f"  {t:<18} {a:+.3f}{'' if a > 0.15 else '   (gated out)'}")
    return oof, corr


def blend(rule_scores, text_scores, w_text=0.35):
    """Combine rules and the distilled text model in rank space, on the rules' scale.

    Two steps, and the second is the one that is easy to get wrong.

    *Combine as ranks.* The two sources are on different scales - the rules emit a
    handful of discrete levels, the ridge a continuous spread - so averaging the values
    lets the rules' coarse quantisation dominate. Ranks are the common currency.

    *Map back onto the rules' scale, by interpolation.* The combined rank is uniform on
    [0, 1] by construction, and using it directly as a BCE target would tell the model
    that exactly half of all knees have a fracture. So it is read back through the rule
    scores - but through their *distinct levels*, interpolating between them, not
    through the sorted array.

    Snapping to the sorted array instead is the obvious version and it silently undoes
    the whole exercise. The rule score takes about four distinct values, so most of its
    mass sits in one enormous atom at "silent"; a rank that lands anywhere inside that
    atom snaps back to the same number, and every ordering the text model contributed
    within the silent reports - which is precisely the ordering it was added to supply -
    is quantised away. Interpolating between the levels keeps the prevalence and the
    meaning of 0.5 while letting the text model order the reports inside each level.
    """
    import pandas as pd

    rule_scores = np.asarray(rule_scores, dtype=np.float32)
    r = pd.DataFrame(rule_scores).rank(pct=True, method="average").values
    t = pd.DataFrame(text_scores).rank(pct=True, method="average").values

    # `w_text` may be a scalar or one weight per target - see `distill`, which returns
    # the per-target OOF correlation used to gate it.
    w = np.broadcast_to(np.asarray(w_text, np.float64).ravel(),
                        (rule_scores.shape[1],)) if np.ndim(w_text) else \
        np.full(rule_scores.shape[1], float(w_text))
    combined = (1.0 - w) * r + w * t

    out = np.empty_like(rule_scores)
    n = len(rule_scores)
    for j in range(rule_scores.shape[1]):
        levels, counts = np.unique(rule_scores[:, j], return_counts=True)
        if len(levels) == 1:
            out[:, j] = levels[0]
            continue
        # Anchor each distinct level at the midpoint of the rank interval it occupies,
        # then interpolate. Anchoring at the midpoint rather than an edge keeps the map
        # centred on the level, so a report the rules scored at that level and the text
        # model had no opinion about comes back out where it went in.
        mid = (np.cumsum(counts) - counts / 2.0) / n
        out[:, j] = np.interp(combined[:, j], mid, levels)
    return out

## 3. Reading the acquisition

`train_series.csv` gives each series a plane and two flags, `Fluid_Sensitive` and
`Fat_Suppression`. The EDA notebook found those two columns are **byte-identical across
all 24,371 rows**.

They name two physically independent things. *Fluid sensitivity* is a property of the
contrast weighting, set by $T_R$ and $T_E$:

$$\text{weighting} = \begin{cases}
T_1 & T_R \lesssim 800\,\text{ms}\\
T_2 & T_R \gtrsim 800\,\text{ms},\ T_E \gtrsim 60\,\text{ms}\\
\text{PD} & T_R \gtrsim 800\,\text{ms},\ T_E \lesssim 60\,\text{ms}
\end{cases}$$

*Fat suppression* is a preparation applied on top of any weighting — a chemically
selective pulse, STIR, or water excitation — and it is what makes marrow oedema
conspicuous. Two identical columns carry one bit between them, not two, so recovering
both axes from the DICOM header is the difference between six slots and three.

Two string-matching traps, both of which silently invert the answer:

- underscore is a word character, so a token test for `we` (water excitation) never
  fires inside `t2_de3d_we_tra`. Separators must be normalised first.
- `ScanOptions` must be matched as **exact tokens**: one vendor writes `SAT_GEMS` for
  *spatial* saturation, so a substring test for `SAT` marks non-suppressed series as
  suppressed.

| slot | plane | weighting | fat sat | what it carries |
|---|---|---|---|---|
| `SAG_FLUID_FS` | sagittal | PD / T2 | yes | meniscal tears, marrow oedema, effusion |
| `COR_FLUID_FS` | coronal | PD / T2 | yes | collateral ligaments, meniscal body |
| `AX_FLUID_FS` | axial | PD / T2 | yes | patellofemoral joint, synovium |
| `SAG_FLUID_NOFS` | sagittal | PD / T2 | no | meniscal morphology at high CNR |
| `COR_T1` | coronal | T1 | no | marrow architecture, bone outline |
| `SAG_T1` | sagittal | T1 | no | anatomy, chronic change |

In [5]:
"""RSNA knee: study-level 12-label pipeline.

Synthesised from two community notebooks, with the parts that were wrong or missing in
both replaced. What is inherited, and from where:

  from `rsna-knee-baseline-v1`   header-recovered slot scheme, constant-physical-scale
                                 sampling, laterality normalisation, uint8 cache,
                                 per-diagnosis slot attention, report-hash grouping
  from `rsna-knee-eda-to-2-5d`   protocol-only features as a legal test-time signal

What is new here is listed in `README.md`; the five that change results are grouped
K-fold instead of a single holdout, gold studies held out of their own fold so the
annotated reference is honest, anatomy-preserving augmentation, a backbone that refuses
to train from random initialisation silently, and rank fusion of the folds and the
protocol model.
"""

from __future__ import annotations

import os

for _v in ("OMP_NUM_THREADS", "OPENBLAS_NUM_THREADS", "MKL_NUM_THREADS"):
    os.environ.setdefault(_v, "4")

import gc
import hashlib
import re
import time
import warnings
from concurrent.futures import ThreadPoolExecutor
from pathlib import Path

import numpy as np
import pandas as pd
import pydicom
import torch
import torch.nn as nn
import torch.nn.functional as F


T0 = time.time()


def log(msg):
    print(f"[{time.time() - T0:7.1f}s] {msg}", flush=True)

In [6]:
class CFG:
    seed = 2026               # folds, target construction, text distillation

    # Separate from `seed` on purpose. This one drives only the stochastic parts of
    # image-model training: head init, batch order, group choice, augmentation. Bump
    # it alone and the folds and the derived targets are byte-identical, so the change
    # in the OOF score is training noise and nothing else. That number is the gate an
    # experiment has to clear before its result means anything.
    train_seed = int(os.environ.get("RSNA_TRAIN_SEED", "2026"))

    # Encoder input, a multiple of the DINOv2 patch size 14. Overridable because `img`
    # and `crop_mm` are two knobs on ONE ratio - what a single token covers in
    # millimetres, 14*crop_mm/img. exp-026 moved that ratio 10.00 -> 8.13 mm/token by
    # shrinking the crop, for free. Raising img moves the same ratio but costs compute
    # quadratically, so the two must be tested separately or the result is confounded:
    #   160mm @ 224 = 10.00 mm/token   (pre-exp-026)
    #   130mm @ 224 =  8.13            (champion, 1.0x compute)
    #   130mm @ 336 =  5.42            (2.25x compute)
    img = int(os.environ.get("RSNA_IMG", "224"))

    # Physical extent of the centre crop. 160 is the historical default and is also
    # exactly the measured MEDIAN field of view of this corpus, which is the worst
    # possible place to stand: the guard below reduces to `FOV > crop_mm`, so a
    # threshold sitting at the median disables the crop on about half the corpus by
    # construction. Measured: 61.1% of slots skipped.
    # Lowering it is the OTHER fix for that defect. Padding (pad_short_fov) grows the
    # image to meet the crop and priced at -0.0012, inside the floor; shrinking the
    # crop makes it apply everywhere instead, and buys a finer pixel pitch for free
    # since pitch = crop_mm / img. Overridable so both fixes share one control.
    crop_mm = float(os.environ.get("RSNA_CROP_MM", "160"))

    # The crop guard `want < min(h, w)` reduces exactly to `FOV > crop_mm`, so a series
    # whose field of view is at or below crop_mm was silently left unnormalised - and
    # the measured median FOV in this corpus is 160 mm, right at the threshold. Padding
    # to crop_mm fixes the scale for those. Off by default so it can be A/B'd against
    # the noise floor rather than smuggled in.
    pad_short_fov = os.environ.get("RSNA_PAD_SHORT_FOV", "0") == "1"

    # Average the last N epochs instead of selecting the best-scoring one.
    # 0 restores the old select-on-the-scored-fold behaviour, so the two are
    # A/B-able. See the note in train_fold for why this exists.
    swa_epochs = int(os.environ.get("RSNA_SWA_EPOCHS", "0"))

    # Whether to accept the geometric laterality fallback (sign of ImagePositionPatient
    # x). Off by default: the tag, ImageLaterality and text sources are all explicit
    # statements by the scanner, while this one is an inference. The run logs how often
    # it agrees with the tag where both exist, so it can be switched on with evidence
    # rather than on faith.
    lat_from_geometry = os.environ.get("RSNA_LAT_GEOMETRY", "0") == "1"
    group = 3                 # slices per encoder input, stacked as the three channels
    # Groups per slot, before the memory budget is applied. Exposed so slice
    # coverage can be priced DOWNWARD: if 1 group (3 slices) scores like 3 groups
    # (9 slices), the curve is already flat and "look at more slices" is dead
    # before it is built.
    n_group_max = int(os.environ.get("RSNA_N_GROUP_MAX", "3"))
    cache_budget_gb = 12.0
    ram_fraction = 0.45       # ceiling on the cache as a share of free RAM

    hdr_threads = 16
    pix_threads = 12

    n_folds = 5
    max_folds_to_run = int(os.environ.get("RSNA_FOLDS", "5"))

    # 12 was inherited from the first version of this pipeline and has never been varied
    # in ~15 experiments, all of which changed how the PICTURES are prepared. The public
    # 0.891 notebook trains 30 (entry 037). Undertraining would be invisible to every
    # experiment run so far, because it affects control and arm identically.
    epochs = int(os.environ.get("RSNA_EPOCHS", "12"))
    # 0 = OneCycle (the historical schedule). >0 = cosine warm restarts of this length.
    cycle_epochs = int(os.environ.get("RSNA_CYCLE_EPOCHS", "0"))
    batch_studies = 8
    lr_backbone = 8e-6        # the encoder is being adapted, not retrained
    lr_head = 1e-3
    weight_decay = 0.02
    unfreeze_last = 6         # trainable transformer blocks, from the output end
    eval_batch = 12
    time_budget = float(os.environ.get("RSNA_TIME_BUDGET", 8.0 * 3600))

    w_text = 0.35             # weight of the distilled text model in the label blend
    w_protocol = 0.10         # weight of the protocol-only model in the final fusion
    gold_weight = 3.0


# Six slots: three planes crossed with the acquisition axes. The fat-suppressed
# fluid-sensitive series exist for nearly every study; the T1 and the non-suppressed
# fluid-sensitive series are scarcer, which is what the presence mask is for.
SLOTS_RECOVERED = [
    ("SAG_FLUID_FS", "Sagittal", True, True),
    ("COR_FLUID_FS", "Coronal", True, True),
    ("AX_FLUID_FS", "Axial", True, True),
    ("SAG_FLUID_NOFS", "Sagittal", True, False),
    ("COR_T1", "Coronal", False, False),
    ("SAG_T1", "Sagittal", False, False),
]

# The alternative: plane x the provided flag, ignoring the recovered weighting.
#
# Worth stating why the recovered scheme is the default. The EDA notebook found that
# `Fluid_Sensitive` and `Fat_Suppression` are byte-identical across all 24,371 series
# rows. They name two physically independent properties - contrast weighting, set by
# TR/TE, and a fat-suppression preparation applied on top of any weighting - so two
# identical columns carry one bit between them, not two. Recovering both axes from the
# DICOM header is therefore not a refinement; it is the difference between six slots
# and three.
SLOTS_PUBLIC = [
    ("SAG_FLUID", "Sagittal", None, True),
    ("COR_FLUID", "Coronal", None, True),
    ("AX_FLUID", "Axial", None, True),
    ("SAG_STRUCT", "Sagittal", None, False),
    ("COR_STRUCT", "Coronal", None, False),
    ("AX_STRUCT", "Axial", None, False),
]

SLOT_SCHEME = os.environ.get("SLOT_SCHEME", "recovered")
SLOTS = SLOTS_PUBLIC if SLOT_SCHEME == "public" else SLOTS_RECOVERED
N_SLOT = len(SLOTS)

FATSAT_OPTS = {"FS", "FATSAT", "FAT_SAT", "FSAT"}
_SEP = re.compile(r"[_\-.]")
_FATSAT_RX = re.compile(r"\bfs\b|fatsat|fat sat|\bstir\b|\bspair\b|\bspir\b|\bwe\b|"
                        r"water excit|\btirm\b|\bsting\b|\bfatsup\b")
_T1_RX = re.compile(r"\bt1\b|\bt1w\b")
_T2_RX = re.compile(r"\bt2\b|\bt2w\b")
_PD_RX = re.compile(r"\bpd\b|\bpdw\b|proton|\bdp\b|dens")


def seed_all(seed=CFG.seed):
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

In [7]:
def find_root(explicit=None):
    if explicit is not None:
        p = Path(explicit)
        if (p / "test.csv").is_file():
            return p
        raise FileNotFoundError(f"no test.csv under {p}")
    for c in [Path("/kaggle/input/competitions/rsna-knee-abnormality-detection"),
              Path("/kaggle/input/rsna-knee-abnormality-detection"),
              Path("data"), Path(".")]:
        if (c / "test.csv").is_file() and (c / "test_series").is_dir():
            return c
    base = Path("/kaggle/input")
    if base.is_dir():
        # last resort: two-level scan, because the mount is sometimes nested one deeper
        for depth1 in sorted(p for p in base.iterdir() if p.is_dir()):
            for cand in [depth1] + sorted(p for p in depth1.iterdir() if p.is_dir()):
                if (cand / "test.csv").is_file():
                    return cand
    raise FileNotFoundError("competition mount not found")


def find_dinov2(variant="small"):
    """Locate a mounted DINOv2 checkpoint directory by variant name."""
    base = Path("/kaggle/input")
    if not base.is_dir():
        return None
    hits = []
    for root, dirs, files in os.walk(base):
        dirs[:] = [d for d in dirs if d not in ("train_series", "test_series")]
        if "config.json" in files and "dinov2" in root.lower():
            hits.append(Path(root))
    for h in hits:
        if variant in str(h).lower():
            return h
    return hits[0] if hits else None

In [8]:
HDR_TAGS = ["SeriesDescription", "SequenceName", "ScanOptions", "ScanningSequence",
            "RepetitionTime", "EchoTime", "Laterality", "PixelSpacing", "Rows",
            "Columns", "RescaleSlope", "RescaleIntercept",
            # Laterality fallbacks. `Laterality` is absent on about half the series,
            # and a study with no side at all is never mirrored - so medial and lateral
            # sit on opposite sides of the image for left and right knees. Four of the
            # twelve targets are side-specific, so that is a third of the metric.
            "ImageLaterality", "StudyDescription", "BodyPartExamined",
            "PatientPosition", "ImagePositionPatient"]


def probe(item):
    split, study, series, path = item
    row = {"split": split, "StudyInstanceUID": study, "SeriesInstanceUID": series,
           "dir": path}
    try:
        files = sorted(e.name for e in os.scandir(path) if e.name.endswith(".dcm"))
        row["files"] = files
        row["n_slices"] = len(files)
        if not files:
            return row
        ds = pydicom.dcmread(os.path.join(path, files[len(files) // 2]),
                             stop_before_pixels=True, force=True)
        for t in HDR_TAGS:
            v = getattr(ds, t, None)
            if v is None:
                row[t] = None
            elif isinstance(v, (list, tuple)) or type(v).__name__ == "MultiValue":
                row[t] = "|".join(str(x) for x in v)
            else:
                row[t] = str(v)
    except Exception as exc:
        row["err"] = str(exc)[:120]
    return row


def walk(root, split):
    base = Path(root) / split
    items = []
    if not base.is_dir():
        return pd.DataFrame()
    for study in os.scandir(base):
        if study.is_dir():
            for series in os.scandir(study.path):
                if series.is_dir():
                    items.append((split, study.name, series.name, series.path))
    with ThreadPoolExecutor(max_workers=CFG.hdr_threads) as pool:
        rows = list(pool.map(probe, items))
    return pd.DataFrame(rows)


def annotate(df):
    """Recover fat suppression and pulse-sequence weighting from the header.

    Two string-matching cautions, both of which silently invert the answer if missed:
    underscore is a word character, so a token test for `we` (water excitation) never
    fires inside `t2_de3d_we_tra` unless separators are normalised first; and GE writes
    `SAT_GEMS` for *spatial* saturation, so ScanOptions must be matched as exact tokens
    or non-fat-suppressed series get marked as suppressed.
    """
    if df.empty:
        return df
    for t in HDR_TAGS:
        if t not in df.columns:
            df[t] = None

    desc = (df["SeriesDescription"].fillna("") + " " + df["SequenceName"].fillna(""))
    desc = desc.str.lower().str.replace(_SEP, " ", regex=True)

    opts = df["ScanOptions"].fillna("").str.upper().str.split("|")
    opts_fs = opts.apply(lambda ts: any(t.strip() in FATSAT_OPTS for t in ts))
    df["fatsat"] = desc.str.contains(_FATSAT_RX) | opts_fs

    tr = pd.to_numeric(df["RepetitionTime"], errors="coerce")
    te = pd.to_numeric(df["EchoTime"], errors="coerce")
    gre = df["ScanningSequence"].fillna("").str.upper().str.contains("GR")
    t1 = desc.str.contains(_T1_RX)
    t2 = desc.str.contains(_T2_RX)
    pdw = desc.str.contains(_PD_RX)

    df["weight"] = np.where(t1 & ~t2 & ~pdw, "T1",
                     np.where(t2 & ~pdw, "T2",
                       np.where(pdw, "PD",
                         np.where(gre, "GRE",
                           np.where(tr < 800, "T1",
                             np.where(te > 60, "T2",
                               np.where(tr >= 800, "PD", "UNK")))))))
    df["fluid"] = np.isin(df["weight"], ["PD", "T2"])
    df["px"] = pd.to_numeric(
        df["PixelSpacing"].fillna("").str.split("|").str[0].replace("", np.nan),
        errors="coerce")
    return df


def pick_slots(series_df, plane_map):
    """One series per slot per study.

    Ties are broken toward the stack with the most slices: a thicker stack samples the
    joint more densely, and the three-slice sampler benefits from the margin.
    """
    if series_df.empty:
        return {}
    series_df = series_df.copy()
    series_df["plane"] = series_df["SeriesInstanceUID"].map(plane_map)
    out = {}
    for study, g in series_df.groupby("StudyInstanceUID"):
        chosen = {}
        for name, plane, fluid, fs in SLOTS:
            sel = (g["plane"] == plane) & (g["fatsat"] == fs)
            # fluid=None means "do not condition on weighting" - the public scheme,
            # where the single provided flag stands in for both axes at once.
            if fluid is not None:
                sel = sel & (g["fluid"] == fluid)
            cand = g[sel]
            if len(cand) == 0 and fluid is False:
                # T1 slots are the scarcest; fall back to any non-fat-sat series in the
                # plane before giving up on the slot entirely.
                cand = g[(g["plane"] == plane) & (~g["fatsat"])]
            if len(cand):
                chosen[name] = cand.sort_values("n_slices", ascending=False).iloc[0]
        out[study] = chosen
    return out

## 4. Sampling at a fixed physical scale

A slice of $N\times N$ pixels at spacing $s$ mm/pixel covers $Ns$ millimetres. Both vary
widely here, so a fixed-pixel resize hands the encoder images whose physical scale
differs by about $3\times$, and a meniscus occupies a different number of pixels in
different studies for no anatomical reason.

Cropping to a constant physical extent $L$ first and resampling to $P\times P$ leaves

$$s_\text{eff} = L/P \ \text{mm/pixel}$$

independent of the acquisition — $160\,\text{mm} / 224 = 0.71$ mm/pixel here.

Intensity needs the same treatment: MR has no absolute scale, so each series is
normalised to its own 1st–99th percentile over the whole volume (not per slice, so
slices keep their relative contrast), which removes an offset that would otherwise track
the site.

### Laterality

Four targets are medial/lateral pairs, and medial is defined against the body midline —
so which side of the *image* it falls on depends on which knee was scanned. The
correction differs by plane: coronal and axial mirror under a horizontal flip, but a
sagittal stack does not, because there the medial–lateral direction is the *slice* axis
and what differs is the order the stack traverses the joint. Where `Laterality` is
absent the volume is left alone — a wrong flip is worse than no flip, and the presence
mask lets the head learn how much to trust each slot.

In [9]:
ORDER_TAGS = ["ImagePositionPatient", "ImageOrientationPatient", "SliceLocation",
              "InstanceNumber"]

# Ordering happens on reader threads, so `d[k] += 1` - three bytecodes, not one - needs
# the lock. An undercounted fallback reads as a clean run.
import threading

ORDER_STATS = {"geometry": 0, "slice_location": 0, "instance_number": 0, "filename": 0}
_ORDER_LOCK = threading.Lock()

# The pixel path had no instrumentation at all, so three separate silent failures were
# invisible: a physical crop that skips itself, a slice that fails to decode and is
# substituted with black, and an inverted photometric interpretation. Each one degrades
# an image without raising anything. Counted here, they become measurements.
PIX_STATS = {"crop_applied": 0, "crop_padded": 0, "crop_skipped_short_fov": 0,
             "crop_no_spacing": 0, "decode_failed": 0, "mono1_inverted": 0,
             "slices_read": 0}
_PIX_LOCK = threading.Lock()


def _bump(key):
    with _ORDER_LOCK:
        ORDER_STATS[key] += 1


def _bump_pix(key, n=1):
    with _PIX_LOCK:
        PIX_STATS[key] += n


def order_slices(directory, files):
    """Return `files` sorted by position along the stack, nearest-first.

    The filenames in this corpus are SOP Instance UIDs, which are random, so the sorted
    filename order is a random permutation of the anatomy. Measured on the competition
    data, filename order agrees with physical order on about 5% of slices.

    That is not a cosmetic problem. Two things downstream assume the list is ordered:
    the sampling window ("the central portion of the stack") and the grouping of
    adjacent slices into the encoder's three channels. Under a random permutation the
    window selects a uniform random sample of the whole series rather than its centre,
    and a group of three adjacent entries is three unrelated positions in the knee - so
    the 2.5D input carried no depth information at all.

    Fallback chain, because not every series records the same tags:
      1. project ImagePositionPatient onto the normal of ImageOrientationPatient. This
         is the only one that is correct for an obliquely angled stack.
      2. SliceLocation, which the scanner has already projected but which is signed
         inconsistently between vendors.
      3. InstanceNumber, acquisition order - right for most stacks, wrong for
         interleaved acquisitions.
      4. filename, i.e. give up, and say so in the counters.
    """
    if len(files) < 2:
        return list(files)

    positions, locations, instances = [], [], []
    normal = None
    for name in files:
        try:
            ds = pydicom.dcmread(os.path.join(directory, name), stop_before_pixels=True,
                                 force=True, specific_tags=ORDER_TAGS)
        except Exception:
            positions.append(None)
            locations.append(None)
            instances.append(None)
            continue
        pos = getattr(ds, "ImagePositionPatient", None)
        orient = getattr(ds, "ImageOrientationPatient", None)
        if normal is None and orient is not None and len(orient) == 6:
            try:
                row_dir = np.array([float(v) for v in orient[:3]])
                col_dir = np.array([float(v) for v in orient[3:]])
                normal = np.cross(row_dir, col_dir)
            except Exception:
                normal = None
        try:
            positions.append(np.array([float(v) for v in pos]) if pos is not None else None)
        except Exception:
            positions.append(None)
        loc = getattr(ds, "SliceLocation", None)
        locations.append(float(loc) if loc is not None else None)
        num = getattr(ds, "InstanceNumber", None)
        instances.append(float(num) if num is not None else None)

    def sorted_by(values, stat):
        _bump(stat)
        return [f for _, f in sorted(zip(values, files), key=lambda pair: pair[0])]

    if normal is not None and all(p is not None for p in positions):
        return sorted_by([float(p @ normal) for p in positions], "geometry")
    if all(loc is not None for loc in locations):
        return sorted_by(locations, "slice_location")
    if all(num is not None for num in instances):
        return sorted_by(instances, "instance_number")
    _bump("filename")
    return list(files)


# Fraction of the ordered stack each plane is sampled across.
#
# These are wider than the 20-80% window this replaced, and the widening is part of the
# ordering fix rather than a separate change. Before ordering, "the central 60%" of a
# random permutation was a uniform sample of the *whole* series; applying that same
# window to a correctly ordered stack would genuinely discard the outer 40% and quietly
# bundle a coverage reduction into this change. These keep the physical span roughly as
# it was, so the ordering is the only variable that moved.
#
# The per-plane split follows Will's (`wguesdon`) reasoning: the menisci sit at the
# medial and lateral extremes of a sagittal stack and Baker's cysts sit posteriorly on
# an axial one, so those planes need their edges; a coronal stack's useful
# anterior-posterior range is narrower.
PLANE_WINDOW = {"Sagittal": (0.10, 0.90), "Axial": (0.10, 0.90),
                "Coronal": (0.15, 0.85)}


def read_slot(rec, n_slice, out_size, plane=None, group=None):
    """`n_slice` slices from one series, physically ordered, at `out_size` pixels.

    Returns uint8 [n_slice, out, out], normalised per-series to its 1st-99th percentile.
    Percentiles rather than min/max because MR intensity has no absolute scale and one
    bright vessel would otherwise compress the whole dynamic range.

    The slices come out as `n_slice // group` anchors spread across the sampling window,
    each anchor contributing `group` *physically adjacent* slices. That layout is what
    `take_group` slices back out into the encoder's three channels, so a channel triplet
    is a genuine depth neighbourhood - roughly 10 mm of knee at this corpus's slice gaps
    - rather than three arbitrary positions. Spreading all nine slices evenly instead
    would cover more of the joint but hand the encoder three views 20 mm apart, which is
    a slab, not a 2.5D triplet.

    A slice of N pixels at spacing s mm covers Ns millimetres. Both vary widely here, so
    a fixed-pixel resize hands the encoder images whose physical scale differs by about
    3x. Cropping to a constant physical extent first fixes the effective scale at
    crop_mm / img mm per pixel regardless of acquisition.
    """
    files, d, px = rec["files"], rec["dir"], rec["px"]
    n = len(files)
    if n == 0:
        return None
    group = CFG.group if group is None else group

    files = order_slices(d, files)

    # exp-016 measured 3 central slices BEATING 9 spread slices by +0.0086 (4x the
    # 0.0020 floor). With one anchor the code takes the window CENTRE; with three it
    # spreads them to the window edges - so "fewer slices" and "more central slices" are
    # confounded in that result. RSNA_WINDOW narrows the window without changing the
    # count, which separates them.
    lo_f, hi_f = PLANE_WINDOW.get(plane, (0.10, 0.90))
    if os.environ.get("RSNA_WINDOW"):
        lo_f, hi_f = (float(x) for x in os.environ["RSNA_WINDOW"].split(","))
    lo, hi = int(lo_f * (n - 1)), int(hi_f * (n - 1))
    if hi <= lo:
        lo, hi = 0, n - 1

    n_anchor = max(1, n_slice // group)
    anchors = (np.linspace(lo, hi, n_anchor).astype(int) if n_anchor > 1
               else np.array([(lo + hi) // 2]))

    idx = []
    for centre in anchors:
        # Adjacent slices around the anchor, clipped into the series rather than
        # wrapped: a wrap would put the far end of the knee in the same channel stack.
        start = int(np.clip(centre - group // 2, 0, max(0, n - group)))
        idx.extend(range(start, min(start + group, n)))
    while len(idx) < n_slice:
        idx.append(idx[-1])

    planes = []
    for i in idx[:n_slice]:
        try:
            ds = pydicom.dcmread(os.path.join(d, files[int(i)]), force=True)
            a = ds.pixel_array.astype(np.float32)
            sl = float(getattr(ds, "RescaleSlope", 1) or 1)
            ic = float(getattr(ds, "RescaleIntercept", 0) or 0)
            a = a * sl + ic
            # MONOCHROME1 means high value renders as black - the image is inverted
            # relative to MONOCHROME2. Left uncorrected, fluid reads dark in those
            # series while reading bright everywhere else, and Effusion and Synovitis
            # are fluid-bright findings. Invert into the MONOCHROME2 convention.
            if str(getattr(ds, "PhotometricInterpretation", "")).strip() == "MONOCHROME1":
                a = a.max() - a
                _bump_pix("mono1_inverted")
            _bump_pix("slices_read")
        except Exception:
            a = None
            _bump_pix("decode_failed")
        planes.append(a)

    shp = next((p.shape for p in planes if p is not None), None)
    if shp is None:
        return None
    planes = [p if (p is not None and p.shape == shp) else np.zeros(shp, np.float32)
              for p in planes]
    vol = np.stack(planes)

    if px and np.isfinite(px) and px > 0:
        want = int(round(CFG.crop_mm / px))
        h, w = shp
        if 16 < want < min(h, w):
            cy, cx = h // 2, w // 2
            half = want // 2
            vol = vol[:, max(0, cy - half):cy + half, max(0, cx - half):cx + half]
            _bump_pix("crop_applied")
        elif want >= min(h, w):
            # The series covers less than crop_mm, so there is nothing to crop away.
            # Resizing here would set mm/px from the acquisition instead of from
            # crop_mm, which is the one thing the constant crop exists to prevent.
            # Padding to the same physical extent keeps the scale fixed; the border is
            # black because no anatomy was acquired there.
            if CFG.pad_short_fov:
                py, pxd = max(0, want - h), max(0, want - w)
                vol = np.pad(vol, ((0, 0), (py // 2, py - py // 2),
                                   (pxd // 2, pxd - pxd // 2)))
                _bump_pix("crop_padded")
            else:
                _bump_pix("crop_skipped_short_fov")
        else:
            _bump_pix("crop_skipped_short_fov")
    else:
        _bump_pix("crop_no_spacing")

    lo_v, hi_v = np.percentile(vol, [1, 99])
    vol = np.clip((vol - lo_v) / max(hi_v - lo_v, 1e-6), 0, 1)

    t = torch.from_numpy(np.ascontiguousarray(vol)).unsqueeze(0)
    t = F.interpolate(t, size=(out_size, out_size), mode="bilinear", align_corners=False)
    # uint8, not float32: intensity is already normalised into [0, 1] here, so eight bits
    # cost nothing a bilinear resize has not already cost, and the cache is a quarter the
    # size for it.
    return (t.squeeze(0) * 255).round().clamp(0, 255).to(torch.uint8)


# Word-boundary side markers. Deliberately NOT including bare Turkish "sag" (right):
# every sagittal protocol in this corpus is named `sag_...`, so that token would mark
# most of the dataset as right knees. Same family of trap as GE's SAT_GEMS. The
# diacritic form is unambiguous and is matched; the stripped form is not.
_SIDE_RX = [
    ("R", re.compile(r"\b(right|rt|r_?knee|knee_?r|dexter|sağ|derech[ao]|rechts?|"
                     r"droite?|δεξ\w*)\b", re.I)),
    ("L", re.compile(r"\b(left|lt|l_?knee|knee_?l|sinister|sol|izquierd[ao]|links?|"
                     r"gauche|αριστερ\w*)\b", re.I)),
]

LAT_STATS = {"tag": 0, "image_laterality": 0, "text": 0, "geometry": 0, "unknown": 0,
             "geom_agree": 0, "geom_disagree": 0}


def _lat_from_text(*fields):
    """Side from any free-text header field, or None if absent or contradictory."""
    blob = " ".join(str(f) for f in fields if f and str(f).lower() != "nan")
    blob = re.sub(r"[^\w\s]", " ", blob)
    hits = {side for side, rx in _SIDE_RX if rx.search(blob)}
    return hits.pop() if len(hits) == 1 else None


def _lat_from_geometry(ipp):
    """Side from the x coordinate of ImagePositionPatient.

    DICOM's patient coordinate system is LPS: +x runs toward the patient's LEFT. A knee
    is therefore centred at negative x when it is the right knee. This is independent of
    head-first/feet-first, because ImagePositionPatient is already expressed in patient
    coordinates rather than scanner coordinates.

    The threshold exists because a value near zero means the scan is near the midline,
    where the sign carries no information.
    """
    try:
        x = float(str(ipp).split("|")[0])
    except (TypeError, ValueError):
        return None
    if abs(x) < 20.0:
        return None
    return "R" if x < 0 else "L"


def resolve_laterality(g):
    """Study -> 'L'/'R'/None, from the strongest available evidence.

    Order: the explicit tag, then ImageLaterality, then side words in any description
    field, then the scan geometry. Each source is counted so the log says how much of
    the corpus each one actually rescued, and the geometric rule is scored against the
    explicit tag wherever both exist - a fallback nobody has validated is a guess.
    """
    vals = [str(x).strip().upper() for x in g.get("Laterality", pd.Series(dtype=object))
            .dropna()]
    vals = [v[0] for v in vals if v and v[0] in ("L", "R")]

    geom = next((s for s in (_lat_from_geometry(v)
                             for v in g.get("ImagePositionPatient",
                                            pd.Series(dtype=object)).dropna()) if s), None)
    if vals and geom:
        _bump_lat("geom_agree" if geom == vals[0] else "geom_disagree")
    if vals:
        _bump_lat("tag")
        return vals[0]

    ivals = [str(x).strip().upper() for x in
             g.get("ImageLaterality", pd.Series(dtype=object)).dropna()]
    ivals = [v[0] for v in ivals if v and v[0] in ("L", "R")]
    if ivals:
        _bump_lat("image_laterality")
        return ivals[0]

    txt = _lat_from_text(*g.get("SeriesDescription", pd.Series(dtype=object)).tolist(),
                         *g.get("StudyDescription", pd.Series(dtype=object)).tolist(),
                         *g.get("BodyPartExamined", pd.Series(dtype=object)).tolist())
    if txt:
        _bump_lat("text")
        return txt

    if geom and CFG.lat_from_geometry:
        _bump_lat("geometry")
        return geom

    _bump_lat("unknown")
    return None


def _bump_lat(key):
    LAT_STATS[key] += 1


def normalise_laterality(img, plane, lat):
    """Map every knee onto a left-knee convention.

    Four of the twelve targets are medial/lateral pairs, and medial is defined against
    the body midline, so which side of the *image* it falls on depends on which knee was
    scanned. Coronal and axial views mirror under a horizontal flip. Sagittal stacks do
    not - each slice is unchanged by mirroring, what differs is the direction the stack
    traverses the joint - so the slice order is reversed instead.

    Where `Laterality` is absent the volume is left alone: a wrong flip is worse than no
    flip, and the presence mask lets the head learn how much to trust each slot.
    """
    if lat != "R":
        return img
    if plane in ("Coronal", "Axial"):
        return torch.flip(img, dims=[-1])
    return torch.flip(img, dims=[0])


def available_ram_gb():
    """Physical RAM available to this process, in GB, or None if it cannot be read."""
    try:
        import psutil
        return psutil.virtual_memory().available / 1024 ** 3
    except Exception:
        pass
    try:                                   # Linux without psutil, which includes Kaggle
        with open("/proc/meminfo") as fh:
            for line in fh:
                if line.startswith("MemAvailable:"):
                    return int(line.split()[1]) / 1024 ** 2
    except Exception:
        pass
    return None


def plan_cache(n_study):
    """Choose how many slices per slot memory allows.

    The cache is n_study x n_slot x slices x img^2 bytes. It grows with the *square* of
    resolution and only linearly with slices, so coverage is the cheap axis and
    resolution the expensive one; when the budget binds it is the slice count that gives
    way. Deciding once, from the training corpus size, keeps train and test on the same
    group layout.

    The configured budget is a ceiling, not a promise. At full corpus size the cache is
    around 12 GB, which fits a 30 GB Kaggle instance and does not fit a 13 GB one - and
    the failure mode is a SIGKILL partway through the decode, which no `try` will catch
    and which costs the whole run. So the budget is also capped at a fraction of what
    the machine actually reports free, leaving room for the model, the reader queue and
    the test cache.
    """
    budget = CFG.cache_budget_gb
    free = available_ram_gb()
    if free is not None:
        safe = CFG.ram_fraction * free
        if safe < budget:
            log(f"cache budget trimmed {budget:.1f} -> {safe:.1f} GB "
                f"({free:.1f} GB free, keeping {1 - CFG.ram_fraction:.0%} headroom)")
            budget = safe
    per_slice = n_study * N_SLOT * CFG.img * CFG.img
    afford = int(budget * 1024 ** 3 // max(per_slice, 1))
    groups = max(1, min(CFG.n_group_max, afford // CFG.group))
    if groups < CFG.n_group_max:
        log(f"cache budget {budget:.1f} GB allows {groups} group(s) of "
            f"{CFG.group}, not {CFG.n_group_max}")
    return groups


def build_cache(slot_map, plane_map, lat_map, tag, n_group):
    """Decode every (study, slot) once into an in-memory uint8 array.

    Fine-tuning revisits the same pixels every epoch, and a study is on the order of a
    hundred and fifty files. Reading them from the mount each epoch would make the epoch
    count a function of I/O rather than of learning.

    Reads are issued in bounded chunks: submitting every job at once lets the reader
    threads run arbitrarily far ahead and the completed buffers accumulate without limit.
    """
    cache_slices = CFG.group * n_group
    studies = sorted(slot_map)
    sidx = {s: i for i, s in enumerate(studies)}
    cache = np.zeros((len(studies), N_SLOT, cache_slices, CFG.img, CFG.img), np.uint8)
    mask = np.zeros((len(studies), N_SLOT), np.float32)
    log(f"{tag}: cache {cache.shape} = {cache.nbytes / 1024 ** 3:.1f} GB")

    jobs = [(st, k, plane, slot_map[st][name])
            for st in studies
            for k, (name, plane, _, _) in enumerate(SLOTS)
            if name in slot_map[st]]
    log(f"{tag}: decoding {len(jobs)} slot-series")

    # Both stat dicts are module-level, so without this the `test:` line reports
    # train+test cumulatively and its percentages describe neither pass.
    ORDER_STATS.update({k: 0 for k in ORDER_STATS})
    PIX_STATS.update({k: 0 for k in PIX_STATS})

    chunk = 512
    done = 0
    with ThreadPoolExecutor(max_workers=CFG.pix_threads) as pool:
        for c0 in range(0, len(jobs), chunk):
            block = jobs[c0:c0 + chunk]
            imgs = pool.map(lambda j: read_slot(j[3], cache_slices, CFG.img, j[2]), block)
            for (st, k, plane, _), img in zip(block, imgs):
                done += 1
                if img is None:
                    continue
                cache[sidx[st], k] = normalise_laterality(img, plane,
                                                          lat_map.get(st)).numpy()
                mask[sidx[st], k] = 1.0
            if done % 4096 < chunk:
                log(f"  {tag} {done}/{len(jobs)}")
            if time.time() - T0 > CFG.time_budget:
                log(f"  {tag}: time budget reached during decode")
                break
    total = sum(ORDER_STATS.values()) or 1
    log(f"{tag}: slice ordering " + ", ".join(
        f"{k} {v / total:.1%}" for k, v in ORDER_STATS.items() if v))
    if ORDER_STATS["filename"] / total > 0.05:
        warnings.warn(
            f"{ORDER_STATS['filename'] / total:.0%} of series fell back to filename "
            "order, which is SOP UID order and therefore random. Those slots carry no "
            "depth structure.", RuntimeWarning, stacklevel=2)

    n_crop = sum(PIX_STATS[k] for k in
                 ("crop_applied", "crop_padded", "crop_skipped_short_fov",
                  "crop_no_spacing")) or 1
    log(f"{tag}: physical scale " + ", ".join(
        f"{k.replace('crop_', '')} {PIX_STATS[k]} ({PIX_STATS[k] / n_crop:.1%})"
        for k in ("crop_applied", "crop_padded", "crop_skipped_short_fov",
                  "crop_no_spacing") if PIX_STATS[k]))
    log(f"{tag}: slices read {PIX_STATS['slices_read']}, "
        f"decode failures {PIX_STATS['decode_failed']}, "
        f"MONOCHROME1 inverted {PIX_STATS['mono1_inverted']}")
    skipped = PIX_STATS["crop_skipped_short_fov"] / n_crop
    if skipped > 0.05:
        warnings.warn(
            f"{skipped:.0%} of slots had a field of view at or below crop_mm="
            f"{CFG.crop_mm:.0f}mm, so no physical normalisation was applied to them and "
            "their mm/px came from the acquisition. Set RSNA_PAD_SHORT_FOV=1 to pad "
            "instead.", RuntimeWarning, stacklevel=2)
    gc.collect()
    return studies, cache, mask

## 5. Aggregating slots into twelve decisions

A study arrives as up to six slot embeddings $x_s$ with a presence mask $m_s$. Pooling
them identically would discard the reason the protocol has three planes at all. Give
every diagnosis $o$ its own query and let it attend over the slots, absent ones masked
out of the softmax:

$$h_s = \phi(x_s) + e_s,\qquad
\alpha_{o,s} = \frac{\exp(\langle h_s, q_o\rangle/\sqrt{H})\, m_s}
{\sum_{s'} \exp(\langle h_{s'}, q_o\rangle/\sqrt{H})\, m_{s'}}$$

The masked softmax renormalises over whatever the study actually contains, so a missing
axial series shifts a diagnosis's attention onto the sequences that are present rather
than feeding it a zero vector.

**The head is deliberately this small.** The label is attached to the *study*, so
nothing in the supervision says which part of it carries the finding. Extra attention
parameters have no signal to learn that from and spend their capacity on noise. Where
the supervision is coarse, the aggregation should be too.

**Why the encoder is trained rather than frozen.** A frozen self-supervised encoder is
the cheap option, and the way it stops paying is informative: resolution, encoder size,
slice coverage and slot aggregation each move the score by less than validation noise.
That pattern is diagnostic — those four axes change how much the model looks and how it
summarises, but none changes the *vocabulary* it looks with. So the last blocks are
adapted, at a learning rate two orders of magnitude below the head's, because the head
is random at initialisation and the encoder starts from a good solution.

In [10]:
class SlotHead(nn.Module):
    """Per-diagnosis attention over the slot embeddings of one study.

    Each finding is read on particular sequences - cruciates sagittally, collateral
    ligaments and the meniscal body coronally, patellar cartilage axially - so pooling
    the slots identically would dilute the one that carries the evidence with the rest.
    The softmax is masked, so a missing axial series shifts a diagnosis's attention onto
    the sequences that are present instead of feeding it a zero vector.

    The aggregation is deliberately this simple. The label is attached to the *study*, so
    nothing in the supervision says which part of a study carries the finding; extra
    attention parameters have no signal to learn that from and spend capacity on noise.
    Where the supervision is coarse, the aggregation should be too.
    """

    def __init__(self, dim, n_slot, n_out, hidden=256, p=0.2):
        super().__init__()
        self.proj = nn.Sequential(nn.LayerNorm(dim), nn.Linear(dim, hidden), nn.GELU())
        self.slot_emb = nn.Parameter(torch.randn(n_slot, hidden) * 0.02)
        self.query = nn.Parameter(torch.randn(n_out, hidden) * 0.02)
        self.drop = nn.Dropout(p)
        self.out = nn.Linear(hidden, n_out)
        self.hidden = hidden

    def forward(self, x, mask):
        h = self.proj(x) + self.slot_emb
        att = torch.einsum("bsh,oh->bos", h, self.query) / self.hidden ** 0.5
        att = att.masked_fill(mask.unsqueeze(1) < 0.5, -1e4).softmax(-1)
        ctx = self.drop(torch.einsum("bos,bsh->boh", att, h))
        return (ctx * self.out.weight.unsqueeze(0)).sum(-1) + self.out.bias


class Encoder(nn.Module):
    """Uniform [B,3,H,W] -> [B,dim] wrapper over either a HF or a timm backbone."""

    def __init__(self, module, kind, dim):
        super().__init__()
        self.module = module
        self.kind = kind
        self.dim = dim

    def forward(self, x):
        if self.kind == "hf":
            out = self.module(pixel_values=x).last_hidden_state
            # CLS and mean-pooled patches: the CLS token carries the global summary and
            # the patch mean the spatially distributed evidence, and the findings here
            # need both.
            return torch.cat([out[:, 0], out[:, 1:].mean(1)], dim=1)
        return self.module(x)

    def blocks(self):
        if self.kind == "hf":
            return list(self.module.encoder.layer)
        for attr in ("blocks", "layers"):
            b = getattr(self.module, attr, None)
            if b is not None:
                return list(b)
        return []


class Model(nn.Module):
    """Encoder plus head, trained end to end.

    A study arrives as a bag of slot images. The bag is flattened for the encoder and
    folded back before the head, so the encoder never sees the study structure and the
    head never sees pixels.
    """

    def __init__(self, encoder):
        super().__init__()
        self.encoder = encoder
        self.head = SlotHead(encoder.dim, N_SLOT, len(TARGETS))
        self.register_buffer("mean", torch.tensor([0.485, 0.456, 0.406]).view(1, 3, 1, 1))
        self.register_buffer("std", torch.tensor([0.229, 0.224, 0.225]).view(1, 3, 1, 1))

    def forward(self, imgs, mask):
        b, s = imgs.shape[:2]
        x = imgs.reshape(b * s, *imgs.shape[2:]).float().div_(255.0)
        x = (x - self.mean) / self.std
        feat = self.encoder(x).reshape(b, s, -1)
        return self.head(feat, mask)


def build_encoder():
    """Resolve a backbone, and refuse to train from random initialisation silently.

    Order: an explicit RSNA_BACKBONE timm name, then a mounted DINOv2 directory, then
    timm with whatever cached weights exist. A frozen self-supervised encoder saturates
    on this task - resolution, encoder size, slice coverage and slot aggregation all
    stop moving the score at the same place, which says the binding constraint is the
    representation, learned as it was on natural images where nothing resembles a torn
    meniscus on a proton-density sequence. So the last blocks are adapted.

    The random-initialisation path stays available but shouts. One of the two source
    notebooks builds `resnet18` with `pretrained=False` and blends 88% of its output
    into the submission, which is a randomly initialised network given two epochs on
    4,407 studies; that is the failure this warning exists to make impossible to miss.
    """
    name = os.environ.get("RSNA_BACKBONE")
    if name:
        import timm
        m = timm.create_model(name, pretrained=True, num_classes=0, global_pool="avg")
        log(f"backbone: timm {name}, dim {m.num_features}")
        enc = Encoder(m, "timm", m.num_features)
    else:
        p = find_dinov2("small")
        if p is not None:
            from transformers import AutoModel
            bb = AutoModel.from_pretrained(str(p))
            enc = Encoder(bb, "hf", bb.config.hidden_size * 2)
            log(f"backbone: DINOv2 from {p}, dim {enc.dim}")
        else:
            import timm
            fallback = "resnet18"
            try:
                m = timm.create_model(fallback, pretrained=True, num_classes=0,
                                      global_pool="avg")
                log(f"backbone: timm {fallback} (pretrained), dim {m.num_features}")
            except Exception as exc:
                m = timm.create_model(fallback, pretrained=False, num_classes=0,
                                      global_pool="avg")
                warnings.warn(
                    "NO PRETRAINED WEIGHTS AVAILABLE - training from random "
                    f"initialisation ({exc}). On 4,407 weakly labelled studies this "
                    "will not learn anything useful. Attach a DINOv2 or timm weights "
                    "dataset before trusting any score from this run.",
                    RuntimeWarning, stacklevel=2)
                log("!! backbone: RANDOM INIT - results are not meaningful")
            enc = Encoder(m, "timm", m.num_features)

    for prm in enc.parameters():
        prm.requires_grad = False
    blocks = enc.blocks()
    if blocks:
        # The early blocks of a self-supervised transformer are generic edge and texture
        # filters. There is not enough supervision here to improve them and quite enough
        # to damage them, so only the last few move.
        for blk in blocks[max(0, len(blocks) - CFG.unfreeze_last):]:
            for prm in blk.parameters():
                prm.requires_grad = True
        if enc.kind == "hf":
            for prm in enc.module.layernorm.parameters():
                prm.requires_grad = True
    else:
        # A CNN exposes no block list to freeze by depth, and its early layers are far
        # cheaper to relearn than a transformer's, so the whole thing is trained.
        for prm in enc.parameters():
            prm.requires_grad = True
    n_tr = sum(p.numel() for p in enc.parameters() if p.requires_grad)
    where = (f"last {min(CFG.unfreeze_last, len(blocks))} of {len(blocks)} blocks"
             if blocks else "all layers (no block structure to freeze by depth)")
    log(f"  trainable: {where}, {n_tr / 1e6:.1f}M params")
    return enc

In [11]:
def augment(imgs):
    """Small in-plane affine plus an intensity scale, applied to a whole bag at once.

    No horizontal flip: laterality was normalised onto a left-knee convention upstream,
    and a horizontal flip would reintroduce exactly the nuisance axis that removed.

    No vertical flip either, which is where this departs from the source notebook. A
    knee coronal or sagittal image has the femur above and the tibia below; flipping it
    top to bottom produces an anatomy that does not exist, and the three OA targets are
    compartment-specific, so the model is being asked to call a finding on a joint whose
    bones have swapped. A rotation of a few degrees is the augmentation that respects
    the acquisition - patient positioning really does vary by that much.
    """
    b = imgs.shape[0]
    dev = imgs.device
    x = imgs.float()
    lead = x.shape[:-3]
    x = x.reshape(-1, *x.shape[-3:])

    ang = (torch.rand(b, device=dev) - 0.5) * (2 * 10 * np.pi / 180)   # +/- 10 degrees
    scale = 1.0 + (torch.rand(b, device=dev) - 0.5) * 0.16             # +/- 8%
    tx = (torch.rand(b, device=dev) - 0.5) * 0.12                      # +/- 6%
    ty = (torch.rand(b, device=dev) - 0.5) * 0.12
    cos, sin = torch.cos(ang) / scale, torch.sin(ang) / scale
    theta = torch.stack([torch.stack([cos, -sin, tx], 1),
                         torch.stack([sin, cos, ty], 1)], 1)           # [b,2,3]

    rep = x.shape[0] // b
    theta = theta.repeat_interleave(rep, 0)
    grid = F.affine_grid(theta, x.shape, align_corners=False)
    x = F.grid_sample(x, grid, mode="bilinear", padding_mode="zeros",
                      align_corners=False)

    gain = 1.0 + (torch.rand(b, 1, 1, 1, device=dev) - 0.5) * 0.2
    x = x * gain.repeat_interleave(rep, 0)
    return x.clamp(0, 255).reshape(*lead, *x.shape[-3:]).to(imgs.dtype)

In [12]:
def protocol_features(hdr, studies):
    """Study-level acquisition features from the header pass.

    These are available at test time - `test_series.csv` and the test DICOM headers both
    exist at inference - so unlike the reports this is a legal feature, not a teacher.
    It carries real signal because protocol tracks indication: a knee scanned with an
    extra fat-suppressed axial series was scanned by someone looking for something.
    """
    if hdr.empty:
        return pd.DataFrame(index=studies).astype(np.float32)
    h = hdr.copy()
    h["slot"] = h["plane"].astype(str).str[0] + "_" + h["fluid"].astype(str) + \
                "_" + h["fatsat"].astype(str)
    f = pd.crosstab(h["StudyInstanceUID"], h["slot"])
    agg = h.groupby("StudyInstanceUID").agg(
        n_series=("SeriesInstanceUID", "size"),
        n_planes=("plane", "nunique"),
        n_fatsat=("fatsat", "sum"),
        n_fluid=("fluid", "sum"),
        med_slices=("n_slices", "median"),
        max_slices=("n_slices", "max"),
        med_px=("px", "median"),
    )
    f = f.join(agg)
    return f.reindex(studies).fillna(0.0).astype(np.float32)


def protocol_model(feat_tr, y_tr, groups, feat_te, n_folds=5):
    """Ridge per target, out-of-fold on the training studies, mean over folds at test."""
    from sklearn.linear_model import Ridge
    from sklearn.pipeline import make_pipeline
    from sklearn.preprocessing import StandardScaler

    cols = sorted(set(feat_tr.columns) & set(feat_te.columns))
    # A slot column present in no training study is a constant, and a constant column
    # makes the normal equations singular rather than merely uninformative.
    cols = [c for c in cols if feat_tr[c].std() > 1e-8]
    if not cols:
        return np.zeros_like(y_tr), np.full((len(feat_te), y_tr.shape[1]), 0.5, np.float32)
    X, Xt = feat_tr[cols].values, feat_te[cols].values
    oof = np.zeros_like(y_tr)
    pred = np.zeros((len(Xt), y_tr.shape[1]), np.float32)
    for f in range(n_folds):
        va = np.flatnonzero(groups == f)
        tr = np.flatnonzero(groups != f)
        if len(va) == 0 or len(tr) < 10:
            continue
        # SVD, not the default Cholesky: the slot-count columns sum exactly to
        # `n_series`, so the design is perfectly collinear by construction and the
        # normal equations are singular however large alpha is.
        m = make_pipeline(StandardScaler(), Ridge(alpha=8.0, solver="svd"))
        m.fit(X[tr], y_tr[tr])
        oof[va] = m.predict(X[va])
        pred += m.predict(Xt) / n_folds
    return oof, pred

In [13]:
def select_device():
    """Pick a device, and prove it can run a kernel before committing the run to it.

    `torch.cuda.is_available()` reports only that a driver and a device exist, not that
    the installed build has code for it. Kaggle's P100 is compute capability 6.0 and the
    pinned PyTorch ships kernels for sm_70 and up, so availability returns True and the
    first real launch dies with `no kernel image is available for execution on the
    device`.

    Left unchecked that happens inside the first training step - which is after the
    header pass and the twenty-minute cache decode, so the run burns most of an hour of
    GPU quota to discover something knowable in five seconds. Hence a real launch here,
    at the top of the run, in the same autocast path training uses.
    """
    if not torch.cuda.is_available():
        log("device: cpu (no CUDA device reported)")
        return torch.device("cpu")

    name = torch.cuda.get_device_name(0)
    major, minor = torch.cuda.get_device_capability(0)
    try:
        x = torch.zeros(8, 3, 32, 32, device="cuda")
        w = torch.zeros(4, 3, 3, 3, device="cuda")
        with torch.autocast("cuda"):
            y = F.conv2d(x, w).flatten(1)
            y = y @ y.T
        y.sum().item()
        torch.cuda.synchronize()
    except Exception as exc:
        arches = " ".join(torch.cuda.get_arch_list())
        raise RuntimeError(
            f"CUDA device '{name}' (sm_{major}{minor}) reports available but cannot "
            f"execute a kernel.\n  underlying error: {exc}\n"
            f"  this torch build has kernels for: {arches}\n"
            "This is the known Kaggle P100 mismatch - the default image's PyTorch "
            "carries no Pascal kernels. Fix: set \"machine_shape\": \"NvidiaTeslaT4\" "
            "in kernel-metadata.json, or choose GPU T4 x2 in the notebook's "
            "accelerator settings. Failing here rather than after the cache build."
        ) from exc

    log(f"device: cuda ({name}, sm_{major}{minor}) - kernel launch verified")
    return torch.device("cuda")


def macro_auc(y, p):
    from sklearn.metrics import roc_auc_score
    return float(np.nanmean([roc_auc_score(y[:, j], p[:, j])
                             if len(set(y[:, j].tolist())) > 1 else np.nan
                             for j in range(y.shape[1])]))


def per_target_auc(y, p):
    from sklearn.metrics import roc_auc_score
    return {t: (roc_auc_score(y[:, j], p[:, j])
                if len(set(y[:, j].tolist())) > 1 else float("nan"))
            for j, t in enumerate(TARGETS)}


# Below this many positives (or negatives) the Hanley-McNeil normal approximation
# stops describing the estimate, so the standard error is reported but not trusted.
MIN_POS_TO_READ = 10


def auc_se(auc, n_pos, n_neg):
    """Hanley-McNeil standard error of an AUC, from the counts alone.

    A low AUC on a rare label is ambiguous: the model may have learned nothing, or
    the estimate may simply be unreadable because there are nine positives. Those
    demand opposite responses, and the mean AUC cannot tell them apart. This makes
    the second case visible without a bootstrap.
    """
    if n_pos < 1 or n_neg < 1 or not np.isfinite(auc):
        return float("nan")
    q1 = auc / (2.0 - auc)
    q2 = 2.0 * auc ** 2 / (1.0 + auc)
    var = (auc * (1 - auc)
           + (n_pos - 1) * (q1 - auc ** 2)
           + (n_neg - 1) * (q2 - auc ** 2)) / (n_pos * n_neg)
    return float(np.sqrt(max(var, 0.0)))


def per_target_report(y, p):
    """Per-label AUC with the positive count and standard error beside it.

    The competition metric is the unweighted mean of twelve AUCs, so every label is
    worth 1/12 regardless of how often it occurs. The mean alone hides which of the
    twelve has headroom and which is already unmeasurable.
    """
    aucs = per_target_auc(y, p)
    out = []
    for j, t in enumerate(TARGETS):
        n_pos, n_neg = int(y[:, j].sum()), len(y) - int(y[:, j].sum())
        auc, se = aucs[t], auc_se(aucs[t], n_pos, len(y) - int(y[:, j].sum()))
        # Two separate reasons a number here may not be actionable, and they are not
        # the same reason. Too few positives means the standard error itself is
        # untrustworthy (the normal approximation needs a real sample, and at n_pos=1
        # it reports a confident interval around nothing). Enough positives but an
        # interval straddling 0.5 means the estimate is sound and the model has
        # genuinely learned nothing. Only the second is worth acting on.
        if n_pos < MIN_POS_TO_READ or n_neg < MIN_POS_TO_READ:
            verdict = f"too few positives (<{MIN_POS_TO_READ}) - se is unreliable"
        elif abs(auc - 0.5) <= 2 * se:
            verdict = "at chance"
        else:
            verdict = "measured"
        out.append({"target": t, "n_pos": n_pos, "n_neg": n_neg,
                    "auc": auc, "se": se, "verdict": verdict})
    return pd.DataFrame(out)


_NORM_PUNCT = re.compile(r"[^\w\s]")
_NORM_NUM = re.compile(r"\d+")
_NORM_WS = re.compile(r"\s+")


def normalise_report(s):
    """Collapse a report to the form its derived labels actually depend on.

    Two reports differing only in punctuation, casing or measurement digits produce the
    same twelve labels, so for fold-assignment purposes they are the same document.
    Hashing the raw string treats them as unrelated and scatters them across folds.
    """
    s = _NORM_PUNCT.sub(" ", str(s).lower())
    return _NORM_WS.sub(" ", _NORM_NUM.sub("#", s)).strip()


def report_fold(report, n_folds):
    """Deterministic fold from the normalised report - no state, no ordering."""
    key = normalise_report(report) or str(report)
    return int(hashlib.md5(key.encode()).hexdigest()[:8], 16) % n_folds


def audit_folds(reports, folds, y, n_folds):
    """Two questions the fold split has to answer before any score off it is believable.

    First: does any group of near-identical reports straddle a fold boundary? If so the
    distilled text model sees a document in training and its twin in validation, and the
    out-of-fold correlation that sets `w_text` is measuring memorisation.

    Second: does every label have enough positives *and* negatives inside every fold?
    AUC is undefined on a single class and unstable near it, and the competition metric
    weights all twelve equally - so one starved label quietly caps the achievable score.
    """
    grp = pd.Series([normalise_report(r) for r in reports])
    span = pd.DataFrame({"g": grp, "f": folds}).groupby("g")["f"].nunique()
    split = int((span > 1).sum())
    if split:
        n_aff = int(grp.isin(span[span > 1].index).sum())
        log(f"  WARNING: {split} near-duplicate report groups span folds "
            f"({n_aff} studies) - text-model OOF is optimistic")
    else:
        log(f"  duplicate check: every near-duplicate report group is inside one fold")

    yb = (np.asarray(y) > 0.5).astype(int)
    starved = []
    for f in range(n_folds):
        va = np.flatnonzero(np.asarray(folds) == f)
        if len(va) == 0:
            continue
        for j, t in enumerate(TARGETS):
            npos = int(yb[va, j].sum())
            nneg = len(va) - npos
            if min(npos, nneg) < MIN_POS_TO_READ:
                starved.append((f, t, npos, nneg))
    if starved:
        log(f"  WARNING: {len(starved)} (fold, label) cells below "
            f"{MIN_POS_TO_READ} positives or negatives:")
        for f, t, npos, nneg in starved[:12]:
            log(f"      fold {f}  {t:<18} pos {npos:>4}  neg {nneg:>4}")
    else:
        log(f"  class balance: all {n_folds} x {len(TARGETS)} cells have "
            f">= {MIN_POS_TO_READ} of each class")
    return {"groups_split": split, "starved_cells": len(starved)}


def take_group(rows, g):
    return rows[:, :, g * CFG.group:(g + 1) * CFG.group]


@torch.no_grad()
def predict(model, cache, mask, idx, dev, n_group):
    """Average the logits over the groups of each slot.

    Training sees one group at a time, which doubles as augmentation along the stack;
    inference averages over all of them.
    """
    model.eval()
    out = []
    for b in range(0, len(idx), CFG.eval_batch):
        sel = idx[b:b + CFG.eval_batch]
        rows = torch.from_numpy(cache[sel]).to(dev)
        m = torch.from_numpy(mask[sel]).to(dev)
        acc = None
        for g in range(n_group):
            with torch.autocast("cuda", enabled=dev.type == "cuda"):
                z = model(take_group(rows, g), m).float()
            acc = z if acc is None else acc + z
        out.append(torch.sigmoid(acc / n_group).cpu().numpy())
    return np.concatenate(out) if out else np.zeros((0, len(TARGETS)), np.float32)


def rank_mean(mats, weights=None):
    """Average several prediction matrices in rank space.

    AUC is invariant under any strictly increasing map of a column, so calibration is
    worth nothing and only order is read. Averaging raw probabilities lets whichever
    model happens to be most confident dominate; averaging ranks combines the only
    information the metric reads.
    """
    mats = [np.asarray(m, np.float64) for m in mats]
    weights = np.ones(len(mats)) if weights is None else np.asarray(weights, np.float64)
    weights = weights / weights.sum()
    acc = np.zeros_like(mats[0])
    for w, m in zip(weights, mats):
        acc += w * pd.DataFrame(m).rank(pct=True).values
    return acc

## 6. Validating without fooling yourself

**Shared reports.** Some reports are byte-identical across studies — a template read for
an unremarkable knee — and every study in such a group gets the same derived target.
Split the group across folds and the model is scored on a target whose source it trained
on. Folds are assigned by a hash of the report text, which keeps duplicates whole.

**Two references, two meanings.** Performance is reported against the derived targets,
which cover every study and measure whether the imaging model learned what the text
says; and against the annotated studies, which are far fewer and measure agreement with
a reading of the *images*. The second resembles the test set. The first has enough
positives per label to tell a real difference from noise.

**The annotated reference has to be held out.** This is where the source notebook leaks:
it scores the annotated AUC over *every* gold study, including the ones in the training
split, and then uses that number to choose the checkpoint. Here gold studies take their
fold assignment like everything else and only held-out gold is scored.

**Choosing an epoch.** The obvious rule — best epoch on the larger reference — is wrong,
because the two measure different things and an epoch that gains on one while losing on
the other has been shown to be *different*, not better. Ranking by

$$\text{score}(e) = \min\big(\mathrm{AUC}^\text{derived}_e,\ \mathrm{AUC}^\text{annot}_e\big)$$

refuses that trade, and makes training length a measured quantity rather than a guess.

In [14]:
def train_fold(cache, mask, Y, W, tr, va, gold_idx, gold_y, n_group, dev, fold):
    """Fit one fold and return (best model, val prediction, chosen-epoch diagnostics).

    Epoch selection reads two references by their *worse* value. They measure different
    things - the derived targets cover every study and say whether the imaging model
    learned what the text says, the annotated subset is far smaller and says whether it
    agrees with a reading of the images - so an epoch that gains on one while losing on
    the other has not been shown to be better, only different. Taking the minimum
    refuses that trade, and makes training length a measured quantity rather than a
    hyperparameter guessed in advance.
    """
    seed_all(CFG.train_seed + fold)
    model = Model(build_encoder()).to(dev)
    enc_params = [p for p in model.encoder.parameters() if p.requires_grad]
    groups = [{"params": model.head.parameters(), "lr": CFG.lr_head}]
    max_lr = [CFG.lr_head]
    if enc_params:
        groups.insert(0, {"params": enc_params, "lr": CFG.lr_backbone})
        max_lr.insert(0, CFG.lr_backbone)
    opt = torch.optim.AdamW(groups, weight_decay=CFG.weight_decay)

    spe = max(len(tr) // CFG.batch_studies, 1)          # optimiser steps per epoch
    steps = max(CFG.epochs * spe, 1)
    if CFG.cycle_epochs > 0:
        # Cosine with warm restarts: the LR is driven back up every cycle_epochs, so the
        # model is repeatedly knocked out of whatever basin it settled in. This is a
        # DIFFERENT question from "train longer" and is kept behind its own flag so the
        # two cannot be confounded - epochs alone answers whether we are undertrained;
        # restarts alone answer whether the schedule shape matters.
        for g, lr in zip(opt.param_groups, max_lr):
            g["lr"] = lr
        sched = torch.optim.lr_scheduler.CosineAnnealingWarmRestarts(
            opt, T_0=CFG.cycle_epochs * spe, T_mult=1)
        log(f"  LR: cosine warm restarts, {CFG.epochs // CFG.cycle_epochs} cycle(s) of "
            f"{CFG.cycle_epochs} epochs")
    else:
        sched = torch.optim.lr_scheduler.OneCycleLR(opt, max_lr=max_lr, total_steps=steps,
                                                    pct_start=0.15)
    scaler = torch.amp.GradScaler("cuda", enabled=dev.type == "cuda")

    yv = (Y[va] > 0.5).astype(int)
    best, best_state, best_ep = -1.0, None, -1
    swa_state, swa_n = None, 0
    stepped = 0

    for ep in range(CFG.epochs):
        model.train()
        perm = np.random.permutation(tr)
        tot, nstep = 0.0, 0
        for b in range(0, len(perm) - CFG.batch_studies + 1, CFG.batch_studies):
            sel = perm[b:b + CFG.batch_studies]
            rows = torch.from_numpy(cache[sel]).to(dev)
            g = int(torch.randint(n_group, (1,)).item())
            imgs = augment(take_group(rows, g))
            m = torch.from_numpy(mask[sel]).to(dev)
            y = torch.from_numpy(Y[sel]).to(dev)
            w = torch.from_numpy(W[sel]).to(dev)
            with torch.autocast("cuda", enabled=dev.type == "cuda"):
                loss = (F.binary_cross_entropy_with_logits(
                    model(imgs, m), y, reduction="none") * w).mean()
            opt.zero_grad(set_to_none=True)
            scaler.scale(loss).backward()
            scaler.step(opt)
            scaler.update()
            if stepped < steps:
                sched.step()
                stepped += 1
            tot += loss.item()
            nstep += 1

        pv = predict(model, cache, mask, va, dev, n_group)
        d_auc = macro_auc(yv, pv)

        # The annotated reference is evaluated on the gold studies *in this fold's
        # validation split only*. Scoring it over every gold study - which is what the
        # source notebook does - measures the model on rows it just trained on, and that
        # number then chooses the checkpoint.
        g_auc = float("nan")
        gv = np.intersect1d(gold_idx, va)
        if len(gv) >= 8:
            gy = gold_y[np.searchsorted(gold_idx, gv)]
            g_auc = macro_auc(gy, predict(model, cache, mask, gv, dev, n_group))

        score = d_auc if not np.isfinite(g_auc) else min(d_auc, g_auc)
        log(f"  fold {fold} epoch {ep + 1}/{CFG.epochs}  loss {tot / max(nstep, 1):.4f}"
            f"  derived {d_auc:.4f}  annot {g_auc:.4f}  worse-of-two {score:.4f}")
        if score > best:
            best, best_ep = score, ep
            best_state = {k: v.detach().cpu().clone()
                          for k, v in model.state_dict().items()}

        # Averaging the LAST `swa_epochs` checkpoints, rather than selecting the one that
        # scored best. Measured cost of selecting: two runs of identical config and
        # identical seed restored epoch 12 and epoch 5 on the same fold and landed 0.0120
        # apart on the pooled OOF - larger than any effect we are trying to detect, which
        # made the whole log unreadable.
        #
        # Two defects, one change. Selecting on the fold we then report is marking our own
        # homework: twelve attempts, best one published. And the coin-flip between two
        # near-equal epochs is pure variance. An average of the tail has neither - it looks
        # at no score at all, so there is nothing to select on, and averaging shrinks the
        # jitter instead of amplifying it.
        #
        # Weight-space averaging is safe here specifically because nothing in this model
        # carries running statistics: DINOv2 and SlotHead are LayerNorm throughout, no
        # BatchNorm, so there are no buffers that would need re-estimating afterwards.
        if CFG.swa_epochs > 0 and ep >= CFG.epochs - CFG.swa_epochs:
            # .clone() is load-bearing: .cpu()/.float() are NO-OPS on a tensor that
            # is already CPU float32, so without it `swa_state` aliases the live
            # parameters and the in-place += below corrupts the model mid-training.
            # It survives on GPU only because .cpu() happens to copy there.
            sd = {k: v.detach().float().cpu().clone()
                  for k, v in model.state_dict().items()}
            if swa_state is None:
                swa_state, swa_n = sd, 1
            else:
                swa_n += 1
                for k in swa_state:
                    swa_state[k] += (sd[k] - swa_state[k]) / swa_n
        if time.time() - T0 > CFG.time_budget:
            log("  time budget reached inside fold")
            break

    if CFG.swa_epochs > 0 and swa_state is not None and swa_n > 1:
        model.load_state_dict({k: v.to(next(model.parameters()).dtype)
                               for k, v in swa_state.items()})
        pv = predict(model, cache, mask, va, dev, n_group)
        sc = macro_auc(yv, pv)
        log(f"  fold {fold}: averaged last {swa_n} epochs, NO selection "
            f"(derived {sc:.4f}; best single epoch was {best_ep + 1} at {best:.4f})")
        return model, pv, sc
    if best_state is not None:
        model.load_state_dict(best_state)
    log(f"  fold {fold}: restored epoch {best_ep + 1} (worse-of-two {best:.4f})")
    return model, predict(model, cache, mask, va, dev, n_group), best

In [15]:
def weights_fingerprint(n_group):
    """Everything a checkpoint's weights silently assume about their input.

    Loading weights into a model whose preprocessing differs does not raise. The shapes
    still line up, `load_state_dict` is happy, and the run produces a perfectly plausible
    submission that is quietly worse - the same failure family as the vacuous scale test
    and the flag nothing read. So the assumptions travel WITH the numbers and are checked
    on the way back in.

    Deliberately excluded: anything that cannot change what a trained model expects to be
    shown - fold count, seeds, time budget, which targets it was trained on.
    """
    return {
        "img": CFG.img,
        "crop_mm": CFG.crop_mm,
        "group": CFG.group,
        "n_group": int(n_group),
        "window": os.environ.get("RSNA_WINDOW", "default"),
        "pad_short_fov": CFG.pad_short_fov,
        "lat_from_geometry": CFG.lat_from_geometry,
        "slots": [s[0] for s in SLOTS],
        "targets": list(TARGETS),
        "backbone": os.environ.get("RSNA_BACKBONE", "dinov2"),
    }


def save_weights(model, fold, n_group, out_dir):
    """Write one member. The filename carries the ARM, not just the fold.

    The fingerprint deliberately excludes what a model was trained ON, because that
    cannot change what pixels it expects - two arms trained on different targets share
    one decode and are meant to load together. But that also means the fingerprint
    cannot tell them apart, so if both write `fold0.pt` into one dataset, one silently
    replaces the other and the "blend" is one arm counted twice. The tag is the only
    thing standing between us and that, so it is part of the name.
    """
    tag = os.environ.get("RSNA_WEIGHTS_TAG", "m")
    p = Path(out_dir)
    p.mkdir(parents=True, exist_ok=True)
    f = p / f"{tag}_fold{fold}.pt"
    if f.exists():
        raise SystemExit(f"REFUSING to overwrite {f.name}. Two members would collide - "
                         f"set RSNA_WEIGHTS_TAG to something distinct for this arm.")
    torch.save({"model": {k: v.detach().cpu() for k, v in model.state_dict().items()},
                "fingerprint": weights_fingerprint(n_group), "fold": int(fold),
                # Provenance: not checked on load (it cannot change what the model
                # expects to be shown), but it makes a mixed dataset self-describing -
                # otherwise ten .pt files are ten anonymous blobs.
                "provenance": {"tag": tag, "train_seed": CFG.train_seed,
                               "targets_from": os.environ.get("RSNA_LLM_LABELS", "rules"),
                               "folds": CFG.max_folds_to_run}}, f)
    log(f"  saved {f.name} ({f.stat().st_size / 1e6:.0f} MB)")


def load_weights_into(model, ckpt_path, n_group):
    """Load one checkpoint, refusing anything that does not match exactly.

    Two refusals, because each catches a different silent failure:
      fingerprint  - the weights were trained on differently prepared images
      strict load  - the architecture moved underneath the numbers; without this a
                     renamed layer loads as random initialisation and still predicts
    """
    ck = torch.load(ckpt_path, map_location="cpu", weights_only=False)
    want, got = weights_fingerprint(n_group), ck.get("fingerprint", {})
    diff = {k: (got.get(k, "<missing>"), v) for k, v in want.items() if got.get(k) != v}
    if diff:
        raise SystemExit(
            f"REFUSING to load {Path(ckpt_path).name}: preprocessing does not match the "
            f"weights.\n" + "\n".join(f"  {k}: checkpoint={a!r} but this run={b!r}"
                                       for k, (a, b) in diff.items()) +
            "\nThese weights were trained on differently prepared images; loading them "
            "would score worse without erroring.")
    missing, unexpected = model.load_state_dict(ck["model"], strict=True)
    if missing or unexpected:
        raise SystemExit(f"REFUSING: state_dict mismatch in {Path(ckpt_path).name} - "
                         f"missing={list(missing)[:5]} unexpected={list(unexpected)[:5]}")
    return ck.get("fold", -1)

In [16]:
def write_submission(study_ids, pred, test_df, path="submission.csv"):
    sub = pd.DataFrame(pred, columns=TARGETS)
    sub.insert(0, "StudyInstanceUID", study_ids)
    sub = test_df[["StudyInstanceUID"]].merge(sub, on="StudyInstanceUID", how="left")
    sub[TARGETS] = sub[TARGETS].fillna(0.5)
    sub.to_csv(path, index=False)
    log(f"{path} {sub.shape}; nulls {int(sub[TARGETS].isna().sum().sum())}")
    return sub


def write_benchmark(root, path="submission.csv"):
    """Write the 0.5 benchmark file immediately.

    A submission that never writes scores nothing at all, which is strictly worse than
    scoring badly. A try/except covers exceptions, but a kill for memory is a SIGKILL and
    never reaches it, so a valid file has to exist from the first second.
    """
    t = pd.read_csv(Path(root) / "test.csv")
    for c in TARGETS:
        t[c] = 0.5
    t.to_csv(path, index=False)

In [17]:
"""Decode the DICOM corpus once and persist the uint8 slot cache to disk.

Why this exists
---------------
`build_cache` already reduces ~500 GB of DICOM to a 4,407 x 6 x 9 x 224 x 224 uint8
array - 11.1 GiB - and then throws it away when the kernel ends. Entry in the pipeline
puts that decode at **55 minutes for 207,801 slices**, and we have paid it on every run
since the baseline. This script pays it once and writes the result out as a Kaggle
Dataset, so training can start from the slab instead of from the mount.

Two consequences, and only the first is certain:
  1. Every later run skips the decode. That is arithmetic, not a hypothesis.
  2. Training becomes portable off Kaggle, because 11 GiB moves and 500 GB does not.
     Whether an off-Kaggle GPU is fast enough is a separate, untested question.

It changes **no pixels**. The array written here is the array `build_cache` would have
returned in RAM, produced by the same call. This is a storage change, not a data change,
and it must never be reported as an experiment - it cannot move a score.

Running it
----------
CPU-only is deliberate. The decode is DICOM reads and resizes; nothing here touches a
GPU, which is what makes it runnable during a GPU-quota lockout (see entry 033/042).

    RSNA_N_GROUP_MAX=3 python tools/build_pixel_cache.py --out /kaggle/working/cache

The config is pinned rather than inferred. `plan_cache` sizes the cache from *available
RAM*, so the same code on a smaller machine silently yields fewer slices - fine for a
throwaway in-RAM cache, fatal for an artifact other runs will consume. Here `n_group`
comes from `RSNA_N_GROUP_MAX` and is asserted, so the cache's shape is a decision and
not an accident of the host.
"""

from __future__ import annotations

import argparse
import json
import os
import sys
import time
from pathlib import Path

import numpy as np
import pandas as pd




# The fields that change the pixels. A training run must refuse a cache whose
# fingerprint differs from its own config, for the same reason the weights loader
# refuses a foreign checkpoint: the failure is otherwise silent and looks like a
# result. `slots` is included because SLOT_SCHEME reorders the slot axis, and an
# axis permutation is invisible in the array shape.
def fingerprint(n_group):
    return {
        "img": CFG.img,
        "crop_mm": CFG.crop_mm,
        "pad_short_fov": CFG.pad_short_fov,
        "lat_from_geometry": CFG.lat_from_geometry,
        "group": CFG.group,
        "n_group": int(n_group),
        # Read inside `read_slot`, not off CFG, so it is easy to forget - and it selects
        # WHICH slices are decoded. Two caches at different windows have identical
        # shapes and different pixels, which is exactly what a fingerprint is for.
        "window": os.environ.get("RSNA_WINDOW", "default"),
        "slots": [s[0] for s in SLOTS],
        "seed": CFG.seed,
    }


def shard_name(tag, i, n):
    """One shard's file stem. Zero-padded so lexical order IS shard order."""
    return tag if n == 1 else f"{tag}.s{i:02d}of{n:02d}"


def shard_studies(slot_map, i, n):
    """Studies belonging to shard `i` of `n`, as a slot_map subset.

    Contiguous blocks of the SORTED study list, because `build_cache` sorts its keys
    internally. Concatenating the shards in index order therefore reproduces exactly
    the array a single unsharded run would have written - the shard boundary is a
    storage detail and must not be a data detail.
    """
    studies = sorted(slot_map)
    bounds = np.linspace(0, len(studies), n + 1).astype(int)
    mine = studies[bounds[i]:bounds[i + 1]]
    return {s: slot_map[s] for s in mine}


def save_split(out, stem, studies, cache, mask):
    """Write one shard as a plain .npy so consumers can memory-map it.

    Uncompressed on purpose. `np.load(..., mmap_mode="r")` lets a training job page
    the slab in from disk and never hold 11 GiB of RAM, which is the difference
    between running on a 16 GB Colab box and not. Compression would save disk and
    take that away.
    """
    np.save(out / f"{stem}_cache.npy", cache)
    np.save(out / f"{stem}_mask.npy", mask)
    pd.DataFrame({"StudyInstanceUID": studies}).to_csv(
        out / f"{stem}_studies.csv", index=False)
    gb = cache.nbytes / 1024 ** 3
    log(f"{stem}: wrote {cache.shape} = {gb:.2f} GiB, "
           f"slot coverage {mask.mean():.1%}")
    return gb


def estimate_decode_time(slot_map, plane_map, lat_map, n_group, n_study, total_slot_series):
    """Decode a small sample, then project the full decode from the measured rate.

    The point is to spend four minutes learning whether the job fits in twelve hours,
    rather than eleven hours learning that it does not. The projection is linear in
    slot-series, which is the right shape - each one is an independent read of a fixed
    number of slices - but it will read optimistically if the sample happens to draw
    thin series, so it is a go/no-go signal and not a schedule.
    """
    studies = sorted(slot_map)[:n_study]
    sub = {s: slot_map[s] for s in studies}
    n_series = sum(len(v) for v in sub.values())
    t0 = time.time()
    build_cache(sub, plane_map, lat_map, "probe", n_group)
    dt = time.time() - t0
    rate = n_series / max(dt, 1e-6)
    proj = total_slot_series / max(rate, 1e-9)
    log(f"probe: {n_series} slot-series in {dt:.1f}s = {rate:.1f}/s")
    log(f"probe: {total_slot_series} slot-series projects to "
           f"{proj / 3600:.2f} h for the full split")
    log(f"probe: shards needed to stay under 10 h each = "
           f"{max(1, int(np.ceil(proj / (10 * 3600))))}")
    return proj


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--out", default="/kaggle/working/cache")
    ap.add_argument("--root", default=None, help="competition mount, autodetected")
    ap.add_argument("--splits", default="train,test")
    ap.add_argument("--shards", type=int, default=1,
                    help="split each split into N independently-runnable shards")
    ap.add_argument("--shard", default="all",
                    help="'all', an index, or a comma list - which shards to build now")
    ap.add_argument("--probe", type=int, default=0,
                    help="decode N studies, project the full time, and exit")
    args = ap.parse_args()

    out = Path(args.out)
    out.mkdir(parents=True, exist_ok=True)
    t0 = time.time()

    root = find_root(args.root)
    log(f"input root: {root}")
    seed_all()

    # The decode is the whole job here, so the default 8 h budget - sized to leave room
    # for training inside a 12 h kernel - would cut the cache short instead of failing.
    # A truncated cache is the dangerous output: it loads, it trains, and the missing
    # studies read as absent slots rather than as an error.
    CFG.time_budget = float(os.environ.get("RSNA_TIME_BUDGET", 11.0 * 3600))

    train_series = pd.read_csv(root / "train_series.csv")
    test_series = pd.read_csv(root / "test_series.csv")
    both = pd.concat([train_series, test_series])
    plane_map = dict(zip(both["SeriesInstanceUID"], both["Anatomical_Plane"]))

    log("header pass: train")
    htr = annotate(walk(root, "train_series"))
    log("header pass: test")
    hte = annotate(walk(root, "test_series"))
    for h in (htr, hte):
        if not h.empty:
            h["plane"] = h["SeriesInstanceUID"].map(plane_map)

    def lat_of(h):
        return {st: resolve_laterality(g)
                for st, g in h.groupby("StudyInstanceUID")} if not h.empty else {}

    slots = {"train": pick_slots(htr, plane_map),
             "test": pick_slots(hte, plane_map)}
    lats = {"train": lat_of(htr), "test": lat_of(hte)}

    # Pinned, not planned. See the module docstring.
    n_group = CFG.n_group_max
    log(f"n_group pinned to {n_group} "
           f"({CFG.group * n_group} slices/slot) from RSNA_N_GROUP_MAX")

    fp = fingerprint(n_group)
    log("fingerprint: " + json.dumps(fp, sort_keys=True))

    if args.probe:
        tag = args.splits.split(",")[0].strip()
        estimate_decode_time(slots[tag], plane_map, lats[tag], n_group, args.probe,
              sum(len(v) for v in slots[tag].values()))
        return

    want = (list(range(args.shards)) if args.shard == "all"
            else [int(x) for x in str(args.shard).split(",")])

    total_gb = 0.0
    written = {}
    for tag in args.splits.split(","):
        tag = tag.strip()
        if not tag or not slots.get(tag):
            log(f"{tag}: no slots, skipped")
            continue
        for i in want:
            stem = shard_name(tag, i, args.shards)

            # Resume. A shard already on disk is a shard already paid for, and the
            # whole point of sharding is that a kernel that dies at hour 11 does not
            # cost the hours before it. Mount the previous run's output as an input
            # and re-run: the finished shards are skipped.
            if (out / f"{stem}_cache.npy").exists():
                log(f"{stem}: present, skipped")
                continue

            sub = shard_studies(slots[tag], i, args.shards)
            expect = sum(len(v) for v in sub.values())
            if not expect:
                # An empty shard is vacuously complete, not missing. `test` holds 3
                # placeholder studies and 4 shards, so one block is empty by
                # construction - and the first real run reported the whole build
                # INCOMPLETE because of it. Record it so the completeness check reads
                # the corpus and not the arithmetic of linspace.
                log(f"{stem}: empty shard (no studies in this block)")
                written[stem] = {"studies": 0, "slot_series": 0}
                continue
            st, C, M = build_cache(sub, plane_map, lats[tag], stem, n_group)

            # `build_cache` breaks out of its decode loop on the time budget and returns
            # a partially filled array. In a training run that costs one run; written to
            # disk it would quietly poison every run that consumed it afterwards. The
            # shard is dropped, not saved, so a resume re-attempts it cleanly.
            got = int(M.sum())
            if got < expect:
                raise RuntimeError(
                    f"{stem}: decoded {got} of {expect} slot-series "
                    f"({got / expect:.1%}). This shard is TRUNCATED and has NOT been "
                    "written. Re-run with more --shards, or raise RSNA_TIME_BUDGET.")
            total_gb += save_split(out, stem, st, C, M)
            written[stem] = {"studies": len(st), "slot_series": got}
            del C, M

    # Merge rather than overwrite. Each resumed kernel builds only the shards it has
    # time for, so a meta file that recorded just this run would erase the record of
    # every shard built before it - and the consumer reads this file to know whether
    # the cache is complete.
    meta_path = out / "cache_meta.json"
    prior = {}
    if meta_path.exists():
        prior = json.loads(meta_path.read_text())
        clash = {k: (prior.get(k), fp[k]) for k in fp
                 if k in prior and prior[k] != fp[k]}
        if clash:
            raise RuntimeError(
                f"config changed between shards: {clash}. Shards built under different "
                "settings are different pixels in one array. Start a fresh --out.")
        # Also fatal, and easier to do by accident: changing --shards between kernels
        # moves the block boundaries, so shard 3 of 8 and shard 3 of 12 are different
        # studies under names that look interchangeable.
        if prior.get("shards", args.shards) != args.shards:
            raise RuntimeError(
                f"--shards changed {prior['shards']} -> {args.shards}. The boundaries "
                "move with it, so existing shards no longer tile the corpus.")
        written = {**prior.get("splits", {}), **written}

    fp["shards"] = args.shards
    fp["splits"] = written
    fp["built_utc"] = time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime())
    fp["decode_seconds"] = round(time.time() - t0, 1) + prior.get("decode_seconds", 0)
    meta_path.write_text(json.dumps(fp, indent=2, sort_keys=True))

    expect_shards = {shard_name(t.strip(), i, args.shards)
                     for t in args.splits.split(",") if slots.get(t.strip())
                     for i in range(args.shards)}
    missing = sorted(expect_shards - set(written))
    log(f"complete: {len(written)}/{len(expect_shards)} shards"
           if not missing else f"INCOMPLETE, still missing: {', '.join(missing)}")

    log(f"done in {(time.time() - t0) / 60:.1f} min, {total_gb:.2f} GiB written")
    # Kaggle's per-notebook output ceiling is the binding limit on this artifact and it
    # is not enormous. Saying so in the log beats discovering it at commit time.
    if total_gb > 15:
        log(f"!! {total_gb:.1f} GiB may exceed the Kaggle output limit - "
               "lower RSNA_N_GROUP_MAX or build the splits as separate datasets")



import sys
sys.argv = ["build_pixel_cache", "--splits", "test,train", "--shards", "4"]
main()


[    0.5s] input root: /kaggle/input/competitions/rsna-knee-abnormality-detection
[    0.7s] header pass: train
[  112.4s] header pass: test
[  144.2s] n_group pinned to 3 (9 slices/slot) from RSNA_N_GROUP_MAX
[  144.2s] fingerprint: {"crop_mm": 130.0, "group": 3, "img": 224, "lat_from_geometry": false, "n_group": 3, "pad_short_fov": false, "seed": 2026, "slots": ["SAG_FLUID_FS", "COR_FLUID_FS", "AX_FLUID_FS", "SAG_FLUID_NOFS", "COR_T1", "SAG_T1"], "window": "0.35,0.65"}
[  144.2s] test.s00of04: empty shard (no studies in this block)
[  144.2s] test.s01of04: cache (1, 6, 9, 224, 224) = 0.0 GB
[  144.2s] test.s01of04: decoding 4 slot-series
[  146.6s]   test.s01of04 4/4
[  146.6s] test.s01of04: slice ordering geometry 100.0%
[  146.6s] test.s01of04: physical scale applied 4 (100.0%)
[  146.6s] test.s01of04: slices read 36, decode failures 0, MONOCHROME1 inverted 0
[  147.0s] test.s01of04: wrote (1, 6, 9, 224, 224) = 0.00 GiB, slot coverage 66.7%
[  147.0s] test.s02of04: cache (1, 6, 9, 